# Hires et al. (2015) from dandiset 000013

Paper. Hires, Gutnisky, Yu, O'Connor & Svoboda (2015), "Low-noise encoding of active touch by
layer 4 in the somatosensory cortex", eLife 4, e06619, [doi:10.7554/eLife.06619](https://doi.org/10.7554/eLife.06619).

Dataset. [DANDI:000013](https://dandiarchive.org/dandiset/000013/0.220126.2143), version
`0.220126.2143` ([doi:10.48324/dandi.000013/0.220126.2143](https://doi.org/10.48324/dandi.000013/0.220126.2143)).
The paper (2015) cites no version of the dandiset, which was created in 2020. This is its only
published version, and so the most recent one; the draft holds the same 52 files (the same
blobs, checked through the API when the attempt was prepared), so no other version could match
the claims file better.

Claims. `claims/000013/10.7554-elife.06619.yaml`, written by a reader from the version of
record (PMC4525079, as served by Europe PMC), its figure legends and figure supplements, and
checked by a second reader. The claims file decides what is tested and how; where it leaves a
method detail open, every reading it lists is run, and where it lists none for a detail that
can change a result, the question is asked in `questions/000013.yaml` and the readings this
notebook can see are all shown.

The data are streamed from the archive; nothing is written to disk. The helper module
`hires.py` holds the loading and every reusable measure, with the source of each parameter.

| Claim | Tested here |
|---|---|
| recordings-counts, recordings-mice | yes |
| behavior-performance, behavior-sample-epoch, behavior-whisking-bouts, behavior-touches-reaction | yes |
| example-cell-rates, example-cell-sample-ff, example-cell-touch-window, example-cell-ff-around-touch, example-cell-phase-modulation | yes |
| text-spikes-per-touch, text-whisking-rates | yes |
| table1-l4-c2, table1-l4-outside, table1-l5, latency-determinable | yes |
| l4-outside-weaker-touch, l5-higher-rates, phase-mod-l4-inside-outside, phase-mod-l4-vs-l5, whisking-rate-l4-no-change, whisking-rate-l5-increase | yes |
| fig3c-touch-spikes-l4c2, fig3c-touch-spikes-outside, fig3c-touch-spikes-l5 | yes |
| fig3d-touch-phase-l4c2, fig3d-touch-phase-outside, fig3d-touch-phase-l5, fig3d-phase-only | yes |
| fig4a-touch-detection-c2, fig4a-touch-detection-outside, fig4b-touch-time-precision, fig4c-phase-decoding | yes |
| fig5c-modulation-index, fig5-modulation-correlations | yes |
| fig6s1b-clusters-less-variable, fig6cd-ff-near-binomial, fig6d-not-refractory | yes |
| fig6e-sample-epoch-ff, fig6e-random-38ms-ff, fig6e-touch-38ms-ff, fig6e-touch-10ms-ff, fig6e-similar-touch-ff | yes |
| whisk-cycle-spikes | yes |

Every claim's data are in the dandiset, so none is blocked before the analysis; the outcomes,
some of them `blocked` as `methods_underspecified` where the readings disagree, are in the
summary at the end.

Answers. A first run asked seven questions (`questions/000013.yaml`). A reader answered them
from the paper, and the claims file now carries the answers. This run follows the updated
criteria: the touch-window cutoff is the transition where the curve levels off, the literal
search from 1 ms being a variant only (fig3c-touch-spikes-l4c2, fig3d-touch-phase-l4c2, and the
claims that use the touch window); the longest window of the cutoff curve is an open detail
with readings of 100 and 200 ms; touch detection adds the reading with every pre-touch window
(fig4a-touch-detection-c2); the time decoder's classes and smoothing are readings
(fig4b-touch-time-precision); the phase decoder uses the cycles of phase tuning, with two
readings for empty bins (fig4c-phase-decoding); the refractory model's rate is raised so that
its output matches the cell's touch response (fig6d-not-refractory); and the L5 whisking
increase is the median of the per-cell differences (whisking-rate-l5-increase).

In [ ]:
import os
import time
import warnings

os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import hires

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
T_START = time.time()

In [ ]:
t0 = time.time()
cells = hires.load_all(max_workers=8)
print(f"loaded {len(cells)} files from dandiset {hires.DANDISET_ID} version {hires.DANDISET_VERSION} in {time.time() - t0:.0f} s")
GROUP_ORDER = ["L4C2", "L4out", "L5"]
by_no = {c.cell_no: c for c in cells}
by_path = {c.path: c.cell_no for c in cells}

## Data check

What was loaded, and each description checked against the values.

In [ ]:
meta = pd.DataFrame([dict(cell=c.cell_no, identifier=c.identifier, subject=c.subject,
                          genotype=c.genotype or "(blank)", region=c.region, layer=c.layer,
                          depth_um=c.depth, group=c.group, trials=len(c.trials),
                          spikes=len(c.spike_t), behaviour_ms=len(c.vt),
                          spike_block=c.spike_block_n[0], spike_offset_ms=1000 * c.extra["spike_offset"],
                          volt_minus_spike_ms=round(1000 * c.volt_offset, 3),
                          spot_check_max_err_s=c.extra["spot_check_err"])
                     for c in cells]).set_index("cell")
print(meta.drop(columns=["identifier"]).to_string())
print("\nfiles", len(meta), "| subjects", meta.subject.nunique(), "| cells per group", meta.group.value_counts().to_dict())
print("trials per file: mean %.1f, s.d. %.1f, min %d, max %d" % (meta.trials.mean(), meta.trials.std(), meta.trials.min(), meta.trials.max()))
print("cell numbers in the identifiers:", sorted(meta.index) == list(range(1, 53)))

In [ ]:
# Units and ranges of every series used, pooled over files
def rng_of(x):
    x = np.asarray(x, float)
    return f"{np.nanmin(x):.3g} to {np.nanmax(x):.3g}"

print("theta_at_base (deg):", rng_of(np.concatenate([c.theta for c in cells])))
print("amplitude (described a.u., read as deg):", rng_of(np.concatenate([c.amp for c in cells])))
print("phase (rad):", rng_of(np.concatenate([c.phase for c in cells])))
print("delta_kappa (a.u.):", rng_of(np.concatenate([c.dkappa for c in cells])))
seglen = np.concatenate([np.diff(np.r_[c.seg_start, len(c.vt)]) for c in cells])
print("behaviour samples per trial:", np.unique(seglen, return_counts=True))
print("spike samples per trial block:", meta.spike_block.value_counts().to_dict(), "(10 kHz)")
pp = pd.concat([c.trials.pole_position for c in cells], ignore_index=True)
print("pole_position range per file:", sorted({(float(c.trials.pole_position.min()), float(c.trials.pole_position.max())) for c in cells}))
print("pole_position, described as mm:", sorted(pp.unique())[:6], "... values per file:",
      sorted({len(c.trials.pole_position.unique()) for c in cells}))
resp = pd.concat([c.trials.assign(cell=c.cell_no) for c in cells], ignore_index=True)
print("trial outcomes:", resp.response.value_counts().to_dict(), "| types:", resp.type.value_counts().to_dict())
print("stim_present values:", resp.stim_present.value_counts().to_dict())
print("outcome against type (rows: type):\n", pd.crosstab(resp.type, resp.response))

In [ ]:
# Touches: every onset should have an offset in its trial; licks against first_lick_time
TCH = {c.cell_no: hires.touches(c) for c in cells}
n_on = sum(len(T.on) for T in TCH.values())
n_inc = sum((~T.complete).sum() for T in TCH.values())
dur = np.concatenate([T.dur for T in TCH.values()])
print(f"touch onsets {n_on}, without an offset in their trial {n_inc}; touch duration ms: median {np.median(dur):.0f}, range {dur.min()} to {dur.max()}, > 100 ms {np.mean(dur > 100):.3f}")
lick_err = []
for c in cells:
    for k, fl in enumerate(c.trials.first_lick_time):
        a = c.seg_start[k]
        b = c.seg_start[k + 1] if k + 1 < len(c.seg_start) else len(c.vt)
        li = np.flatnonzero(c.lick[a:b])
        if np.isnan(fl):
            lick_err.append(np.nan if len(li) == 0 else -1.0)
        else:
            lick_err.append((c.seg_t0[k] + li[0] / 1000 - fl) * 1000 if len(li) else np.inf)
lick_err = np.array(lick_err)
fin = lick_err[np.isfinite(lick_err)]
print(f"first_lick_time against the first lick onset of the lick series: {len(fin)} trials, |difference| median {np.median(np.abs(fin)):.2f} ms, max {np.abs(fin).max():.2f} ms;"
      f" lick series without first_lick_time {np.sum(lick_err == -1)}, first_lick_time without a lick in the series {np.sum(np.isinf(lick_err))}")
pa_first = []
for c in cells:
    for k in range(len(c.trials)):
        a = c.seg_start[k]
        b = c.seg_start[k + 1] if k + 1 < len(c.seg_start) else len(c.vt)
        r = hires.runs(c.pole[a:b])
        pa_first.append(np.nan if len(r) == 0 else (c.seg_t0[k] + r[0, 0] / 1000 - c.trials.pole_in_time.iloc[k]))
pa_first = np.array(pa_first)
print(f"pole_available onset minus pole_in_time (s): median {np.nanmedian(pa_first):.3f}, range {np.nanmin(pa_first):.3f} to {np.nanmax(pa_first):.3f}; trials without pole_available {np.isnan(pa_first).sum()}")

In [ ]:
# The stored amplitude and phase against the Methods' filter (6-60 Hz, 4th-order Butterworth, Hilbert),
# and the stored theta_filt and set_point against their descriptions, on the first 3 trials of every file
from scipy.signal import butter, filtfilt, hilbert
bb, aa = butter(2, [6 / 500, 60 / 500], "bandpass")
chk = []
for c in cells:

    for k in range(3):
        a, b = c.seg_start[k], c.seg_start[k + 1]
        y = filtfilt(bb, aa, c.theta[a:b].astype(float))
        h = hilbert(y)
        chk.append(dict(cell=c.cell_no, amp_r=np.corrcoef(np.abs(h), c.amp[a:b])[0, 1],
                        amp_maxdiff=np.abs(np.abs(h) - c.amp[a:b]).max(),
                        phase_maxdiff=np.abs(np.angle(np.exp(1j * (np.angle(h) - c.phase[a:b])))).max()))
chk = pd.DataFrame(chk)
print("recomputed Hilbert amplitude vs stored, %d trials: r median %.5f, min %.4f; max |diff| median %.3f deg, max %.2f deg" %
      (len(chk), chk.amp_r.median(), chk.amp_r.min(), chk.amp_maxdiff.median(), chk.amp_maxdiff.max()))
# theta_filt, set_point and the voltage, read for the first seconds of every file
SER = hires.check_series_all()
SER["cell"] = [by_path[p] for p in SER.path]
SER = SER.set_index("cell").sort_index()
SER["volt_ptp_mV"] = SER.volt_ptp_stored * SER.volt_conversion * 1000
SER["spike_offset_ms"] = meta.spike_offset_ms
print(SER[["theta_filt_r", "theta_filt_maxdiff", "set_point_r", "volt_ptp_mV", "volt_unit", "spike_offset_ms", "file_name_ogen", "stimulus_groups"]].round(4).to_string())
print(SER.groupby("spike_offset_ms").agg(files=("path", "size"), theta_filt_r_min=("theta_filt_r", "min"), theta_filt_maxdiff=("theta_filt_maxdiff", "max"),
                                         volt_mV_min=("volt_ptp_mV", "min"), volt_mV_max=("volt_ptp_mV", "max")).to_string())
SER["theta_filt_is_bandpass"] = SER.theta_filt_maxdiff < 0.01
SER["volt_in_mV_range"] = SER.volt_ptp_mV > 1
print("files by spike offset (rows) and by whether theta_filt equals amplitude*cos(phase) and the voltage spans more than 1 mV (columns):")
print(pd.crosstab(SER.spike_offset_ms, [SER.theta_filt_is_bandpass, SER.volt_in_mV_range]))
print("files whose set_point holds missing values (r undefined):", int(SER.set_point_r.isna().sum()), "| r where defined: %.3f to %.3f" % (SER.set_point_r.min(), SER.set_point_r.max()))
print("files named 'ogen': %d; with any stimulus series: %d; trials with stim_present = 1: %d" %
      (SER.file_name_ogen.sum(), (SER.stimulus_groups > 0).sum(), int((resp.stim_present == 1).sum())))

Clocks. The trials table, the behaviour series and the spike train each carry timestamps.
Behaviour segments start within 5 ms of each trial's `start_time`. In 36 files the
`SpikeTrain` timestamps start 10 ms before the voltage (`CurrentClampSeries`) timestamps, though
the spike samples line up with the voltage samples one for one; in 16 files the two agree. The
cell below asks which clock puts the spikes on the behaviour's time: the touch response is the
contrast. If the `SpikeTrain` timestamps are right, the touch-response peak should fall at the
same latency in both kinds of file; if the voltage timestamps are right, the peaks in the 36
files should sit 10 ms earlier than the others under the `SpikeTrain` clock.

In [ ]:
rows = []
for c in cells:
    T = TCH[c.cell_no]
    for clock in hires.CLOCKS:
        s = c.spikes_vt(clock)
        t, h, n = hires.pcth(c, s, T.on)
        post = t >= 0
        rows.append(dict(cell=c.cell_no, group=c.group, offset_ms=1000 * c.extra["spike_offset"], clock=clock,
                         peak_ms=int(t[post][np.argmax(h[post])]), peak_rate=h[post].max() * 1000,
                         pre_rate=h[~post].mean() * 1000))
clk = pd.DataFrame(rows)
l4 = clk[clk.group == "L4C2"]
print(l4.groupby(["clock", "offset_ms"]).peak_ms.describe()[["count", "mean", "50%", "min", "max"]])
for clock in hires.CLOCKS:
    x = l4[(l4.clock == clock) & (l4.offset_ms == 0)].peak_ms
    y = l4[(l4.clock == clock) & (l4.offset_ms == -10)].peak_ms
    print(f"{clock} clock: L4 C2 touch peak, files without offset median {x.median():.0f} ms (n={len(x)}), files with the 10 ms offset median {y.median():.0f} ms (n={len(y)}); rank sum p = {stats.mannwhitneyu(x, y).pvalue:.3g}")
print("peak over pre-touch rate, L4 C2 cells, spike clock: median %.1f-fold, min %.1f" %
      ((l4[l4.clock == "spike"].peak_rate / l4[l4.clock == "spike"].pre_rate.clip(lower=1e-9)).median(),
       (l4[l4.clock == "spike"].peak_rate / l4[l4.clock == "spike"].pre_rate.clip(lower=1e-9)).min()))
# Shifted-label control: the same contrast with touch onsets moved by a random amount (200 to 1000 ms) within the trial
rng_clk = np.random.default_rng(11)
ratio_true, ratio_shift = [], []
for c in cells:
    if c.group != "L4C2":
        continue
    T = TCH[c.cell_no]
    s = c.spikes_vt("spike")
    _, h, _ = hires.pcth(c, s, T.on)
    ratio_true.append(h[50:80].mean() / max(h[:50].mean(), 1e-9))
    shifted = T.on + rng_clk.integers(200, 1000, len(T.on)) * rng_clk.choice([-1, 1], len(T.on))
    lo = c.seg_start[c.trial_of[T.on]]
    hi = np.r_[c.seg_start[1:], len(c.vt)][c.trial_of[T.on]] - 1
    shifted = np.clip(shifted, lo, hi)
    _, h2, _ = hires.pcth(c, s, shifted)
    ratio_shift.append(h2[50:80].mean() / max(h2[:50].mean(), 1e-9))
print("L4 C2, rate 0-30 ms after over 50 ms before touch: true onsets median %.2f, shifted onsets median %.2f" %
      (np.median(ratio_true), np.median(ratio_shift)))
CLOCK = "spike"

Recordings deposited twice. Pairs of files from the same subject are compared trial by
trial on the whisker angle and the spike train.

In [ ]:
dups = []
sig = {}
for c in cells:
    th = [c.theta[a:b][:200].tobytes() for a, b in zip(c.seg_start, np.r_[c.seg_start[1:], len(c.vt)])]
    sig[c.cell_no] = (set(th), np.round(c.spike_t, 4))
for i, a in enumerate(cells):
    for b in cells[i + 1:]:
        if a.subject != b.subject:
            continue
        same_theta = len(sig[a.cell_no][0] & sig[b.cell_no][0])
        same_spk = len(np.intersect1d(sig[a.cell_no][1], sig[b.cell_no][1]))
        dups.append(dict(a=a.cell_no, b=b.cell_no, subject=a.subject, trials_a=len(a.trials), trials_b=len(b.trials),
                         identical_theta_trials=same_theta, shared_spike_times=same_spk,
                         spikes_a=len(a.spike_t), spikes_b=len(b.spike_t)))
dups = pd.DataFrame(dups)
print(dups.to_string(index=False))
print("pairs with any identical theta trial:", int((dups.identical_theta_trials > 0).sum()), "of", len(dups))

## Measures shared by the claims

Computed once per cell, on the `SpikeTrain` clock, with every reading the claims file lists.

- Touches: onset and offset series paired within each trial. Exploration time: whisking
  (amplitude > 2.5 deg) or touch, leaving out touches > 100 ms, while the pole is in reach.
- Onset latency: first 1 ms bin of the PCTH 0 to 50 ms after onset above the pre-touch mean
  (-50 to 0 ms) plus two s.d.; from first touches (`lat_first`) and from all touches (`lat_all`).
- Phase tuning: spikes in whisking cycles of exploration time outside touch (cycles with
  amplitude > 2.5 deg throughout, monotonic phase, no lick, not within 70 ms after a touch),
  12-bin rate histogram, cosine fit, depth A / (A + B).
- Rates: whisking (amplitude > 2.5 deg, read with and without touch epochs) and non-whisking
  (>= 100 ms with amplitude < 1.25 deg, no touch, no lick), over all tracked time.
- Minimum ISI: the shortest interval between two spikes of one trial.

In [ ]:
t0 = time.time()
D = {}
base_rows = []
for c in cells:
    T = TCH[c.cell_no]
    s = c.spikes_vt(CLOCK)
    E = hires.exploration_mask(c, T)
    touch = hires.touch_mask(c, T)
    cyc = hires.cycle_table(c, T)
    first = T.on[T.order == 1]
    lat_first = hires.onset_latency(c, s, first)
    lat_all = hires.onset_latency(c, s, T.on)
    phase_cyc = cyc[cyc.phase_ok & cyc.in_sample]
    cen, rate, cnt, occ = hires.phase_tuning(c, s, phase_cyc, E & ~touch)
    A, B, phi, depth = hires.cos_fit(cen, np.nan_to_num(rate))
    if cnt.sum() == 0:
        phi, depth = np.nan, np.nan
    nw = hires.nonwhisk_mask(c, T)
    D[c.cell_no] = dict(T=T, s=s, E=E, touch=touch, cyc=cyc, phi=phi, phase_rate=rate, phase_cnt=cnt, phase_occ=occ)
    base_rows.append(dict(
        cell=c.cell_no, group=c.group, subject=c.subject, n_touch=len(T.on), n_first=len(first),
        lat_first=lat_first, lat_all=lat_all, depth_mod=depth, phi_pref=phi, phase_spikes=int(cnt.sum()),
        phase_time_s=float(occ.sum()),
        rate_whisk_no_touch=hires.rate_in(c, s, hires.whisk_mask(c, T, False)),
        rate_whisk_with_touch=hires.rate_in(c, s, hires.whisk_mask(c, T, True)),
        rate_nonwhisk=hires.rate_in(c, s, nw), nonwhisk_s=nw.sum() / 1000,
        min_isi_ms=hires.isi_min(c), explore_s=E.sum() / 1000, n_spikes=len(s)))
base = pd.DataFrame(base_rows).set_index("cell")
print(f"{time.time() - t0:.0f} s")
print(base.round(3).to_string())
print("cells with fewer than 10 spikes in the phase-analysis cycles:", base[base.phase_spikes < 10].groupby("group").size().to_dict(),
      "| with none (no depth):", base[base.phase_spikes == 0].index.tolist())

### Touch spike windows

The claims file (fig3c-touch-spikes-l4c2) leaves four details open and lists their readings:
the shuffle (`uniform` over exploration time, or `circular` shifts within each trial), the
closest touch onset (`before` the spike, or on `either` side), the 1 ms step, and the longest
window over which the curve is built and the degree-11 polynomial fitted, 100 or 200 ms
(`tmax`, readings r100 and r200). Where the search for the cutoff starts (`start`) was asked,
and the answer is now in the criterion: `cross`, the first increment below chance after the
increment has been above it, the transition where "the curve levels off" (Results, paragraph on
Figure 3). The literal search from t = 1 ms (`first`) is computed as well and shown only as a
variant; the fitted polynomial dips at the start of a steep curve, so it often stops at 1 ms.
Outcomes rest on `cross`.

Cells without a first-touch latency take reading (a), latency 0 ms, or (b), the all-touch
latency where there is one and else 0 ms (claim fig3c-touch-spikes-outside). Cells with a
first-touch latency use it under both. Each cell and reading has its own random generator.

In [ ]:
SEED_WIN = 3000
WIN = [(sh, nr, tm) for sh in ("uniform", "circular") for nr in ("before", "either") for tm in hires.TMAX_READINGS]
t0 = time.time()
win_rows = []
for c in cells:
    d = D[c.cell_no]
    lf, la = base.loc[c.cell_no, "lat_first"], base.loc[c.cell_no, "lat_all"]
    if pd.notna(lf):
        lats = {("a", "b"): float(lf)}
    else:
        lats = {("a",): 0.0, ("b",): float(la) if pd.notna(la) else 0.0}
    for li, (lr, L) in enumerate(lats.items()):
        for k, (sh, nr, tm) in enumerate(WIN):
            rng = np.random.default_rng([SEED_WIN, c.cell_no, k, li])
            res = hires.touch_window(c, d["s"], d["T"], d["E"], L, sh, nr, tm, rng)
            for st, r in res.items():
                for lr1 in lr:
                    win_rows.append(dict(cell=c.cell_no, group=c.group, lat_reading=lr1, shuffle=sh, nearest=nr,
                                         tmax=tm, start=st, latency=L, t_on=r["t_on"], frac_spikes=r["frac_spikes"],
                                         frac_time=r["frac_time"], crossed=r["crossed"], n_explore_spikes=r["n_spikes"]))
W = pd.DataFrame(win_rows)
WKEY = ["lat_reading", "shuffle", "nearest", "tmax", "start"]
print(f"touch windows: {len(W)} rows in {time.time() - t0:.0f} s")
print("cells whose increment never rose above chance under `cross` (the literal cutoff is used):")
print(W[(W.start == "cross") & ~W.crossed].groupby(["group"] + WKEY[1:4]).cell.apply(list).to_string())

In [ ]:
# Spikes per touch in each cell's window, first touches of trials and later touches apart
t0 = time.time()
spt_rows = []
for (no, lr, sh, nr, tm, st), g in W.set_index(["cell"] + WKEY).groupby(level=[0, 1, 2, 3, 4, 5]):
    r = g.iloc[0]
    c = by_no[no]
    d = D[no]
    T = d["T"]
    n = hires.spikes_per_touch(c, d["s"], T, r.latency, r.t_on)
    spt_rows.append(dict(cell=no, lat_reading=lr, shuffle=sh, nearest=nr, tmax=tm, start=st,
                         spt_first=n[T.order == 1].mean() if (T.order == 1).any() else np.nan,
                         spt_later=n[T.order > 1].mean() if (T.order > 1).any() else np.nan))
SPT = pd.DataFrame(spt_rows)
W = W.merge(SPT, on=["cell"] + WKEY)
print(f"{time.time() - t0:.0f} s")

### Touch and phase windows

After the touch spikes are taken out, the remaining exploration spikes are indexed by the time
to the closest preferred-phase time of a whisking cycle (cycles with amplitude > 2.5 deg
throughout, overlapping exploration time, monotonic or not), and a cutoff t_w is found by the
same rule, under the same readings as the touch window (claim fig3d-touch-phase-l4c2). Its open
detail (4), the phase window read as +-t_w (`pm`) or t_w wide in total (`total`), doubles the
readings. The shuffle runs over exploration time outside the touch windows.

In [ ]:
SEED_PH = 4000
t0 = time.time()
ph_rows = []
for c in cells:
    d = D[c.cell_no]
    cyc = d["cyc"]
    Ec = np.r_[0, np.cumsum(d["E"])]
    wc = cyc[cyc.whisking & ((Ec[cyc.stop.to_numpy()] - Ec[cyc.start.to_numpy()]) > 0)]
    sub = W[W.cell == c.cell_no]
    for li, (L, g0) in enumerate(sub.groupby("latency")):
        lrs = sorted(g0.lat_reading.unique())
        g0 = g0[g0.lat_reading == lrs[0]]
        for k, (sh, nr, tm) in enumerate(WIN):
            g = g0[(g0.shuffle == sh) & (g0.nearest == nr) & (g0.tmax == tm)].set_index("start")
            t_on = {st: int(g.loc[st, "t_on"]) for st in ("first", "cross")}
            for wi, width in enumerate(("pm", "total")):
                rng = np.random.default_rng([SEED_PH, c.cell_no, k, li, wi])
                res = hires.phase_window(c, d["s"], d["T"], d["E"], L, t_on, nr, d["phi"], wc, sh, tm, width, rng)
                for st, r in res.items():
                    for lr in lrs:
                        ph_rows.append(dict(cell=c.cell_no, group=c.group, lat_reading=lr, shuffle=sh, nearest=nr, tmax=tm,
                                            start=st, width=width, t_w=r["t_w"], frac_tp=r["frac_spikes"], time_tp=r["frac_time"]))
PH = pd.DataFrame(ph_rows).merge(W[["cell"] + WKEY + ["frac_spikes", "frac_time"]], on=["cell"] + WKEY)
PH["frac_phase_only"] = PH.frac_tp - PH.frac_spikes
PKEY = WKEY + ["width"]
print(f"phase windows: {len(PH)} rows in {time.time() - t0:.0f} s")

In [ ]:
def within(v, exp, tol):
    return bool(np.isfinite(v) and abs(v - exp) <= tol + 1e-9)


def msm(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return dict(n=len(x), mean=x.mean() if len(x) else np.nan, sd=x.std(ddof=1) if len(x) > 1 else np.nan,
                median=np.median(x) if len(x) else np.nan)


def ranksum(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    return stats.ranksums(a, b).pvalue


G = {g: base.index[base.group == g].tolist() for g in GROUP_ORDER}

## recordings-counts

> The study holds 52 single-unit recordings, 31 L4 cells in the C2 barrel, 10 L4 cells outside
> C2 and 11 L5 cells near C2, with 115 +- 60 (mean +- s.d.) trials per neuron.

Groups come from the files' electrode location (`brain_subregion` C2 or surround,
`cortical_layer` 4 or 5). The files mark no subset of trials as analysed, so readings (a), all
trials, and (b), the performing trials analysed, are the same here.

In [ ]:
print(meta.groupby(["region", "layer"]).size())
print("L4 C2 %d, L4 outside C2 %d, L5 %d" % (len(G["L4C2"]), len(G["L4out"]), len(G["L5"])))
print("trials per neuron, all trials (a) = (b): mean %.1f, s.d. %.1f, n = %d" % (meta.trials.mean(), meta.trials.std(), len(meta)))
print("depth by group (um):", meta.groupby("group").depth_um.agg(["min", "max"]).to_dict("index"))
print("within tolerance: counts", [len(G["L4C2"]) == 31, len(G["L4out"]) == 10, len(G["L5"]) == 11],
      "trials", within(meta.trials.mean(), 115, 60))

**Result: reproduced.** The files hold 31 L4 C2, 10 L4 surround and 11 L5 C2 recordings, one per
file, with 115.0 +- 60.6 trials per neuron (mean +- s.d.), the paper's 115 +- 60. The depths fit
the Methods' ranges (L4 405 to 565 um, L5 627 to 844 um).

## recordings-mice

> The 52 recordings come from 21 mice of four lines, 21 neurons from 7 C57BL/6 mice, 6 from 3
> VGAT-ChR2(H134R) mice, 17 from 8 PV-ires-cre mice and 8 from 3 PV-ires-cre x Ai32 mice.

In [ ]:
gt = meta.groupby("genotype").agg(mice=("subject", "nunique"), neurons=("subject", "size"))
print(gt)
print("subjects:", meta.subject.nunique(), "| recordings:", len(meta))
print("subjects whose identifier starts with 'jf':", sorted(meta.subject[meta.subject.str.startswith("jf")].unique()))
print("cells per subject:", meta.groupby("subject").size().to_dict())

**Result: not reproduced.** The dandiset has 52 recordings, as the paper, but 23 subjects, not
21, and its genotypes do not give the paper's counts per line: C57BL6 9 mice and 22 neurons
(paper 7 and 21), Ai32 x PV-Cre 3 mice and 7 neurons (paper 3 and 8), PV-Cre 8 mice and 17
neurons (as the paper), and 3 mice and 6 neurons with a blank genotype, which match the
paper's VGAT-ChR2 counts but carry no label. Two subject identifiers more than the paper's mice
may be one animal under two names (the `jf` and `anm` series), but nothing in the files says so.

## behavior-performance

> Across the 52 sessions the fraction of trials correct was 0.740 +- 0.086 (mean +- s.d.).

Correct trials are hits and correct rejections. Reading (a), all trials, and reading (b), the
dataset's own marking of the trials used, are the same: the files mark none.

In [ ]:
perf = pd.Series({c.cell_no: c.trials.response.isin(["Hit", "CR"]).mean() for c in cells})
print("fraction correct per session: mean %.3f, s.d. %.3f, range %.3f to %.3f, n = %d" % (perf.mean(), perf.std(), perf.min(), perf.max(), len(perf)))
print("within 0.740 +- 0.086:", within(perf.mean(), 0.740, 0.086))
print("by group:", perf.groupby(base.group).mean().round(3).to_dict())

**Result: reproduced.** The fraction of trials correct is 0.741 +- 0.085 over the 52 sessions,
the paper's 0.740 +- 0.086.

## behavior-sample-epoch

> The sample epoch, when the pole is presented, lasted 1.54 to 4.05 s, 2.39 s on average.

The sample epoch is the period `pole_available` is true in a trial (first sample on to last
sample on). Readings (a) over all trials pooled and (b) over per-session means. Trials whose
pole is still in reach when the 4.5 to 5 s recording ends are cut there; they are counted, and
`pole_out_time - pole_in_time` from the trials table is shown as a variant.

In [ ]:
ep_rows = []
for c in cells:
    for k in range(len(c.trials)):
        a = c.seg_start[k]
        b = c.seg_start[k + 1] if k + 1 < len(c.seg_start) else len(c.vt)
        r = hires.runs(c.pole[a:b])
        if len(r) == 0:
            continue
        ep_rows.append(dict(cell=c.cell_no, trial=k, dur=(r[-1, 1] - r[0, 0]) / 1000, n_runs=len(r),
                            cut=bool(c.pole[a] or c.pole[b - 1]), on_ms=r[0, 0], off_ms=r[-1, 1],
                            table=c.trials.pole_out_time.iloc[k] - c.trials.pole_in_time.iloc[k]))
EP = pd.DataFrame(ep_rows)
sess = EP.groupby("cell").dur.mean()
print("trials %d, with more than one run of pole_available %d, cut at the end of the recording %d" % (len(EP), (EP.n_runs > 1).sum(), EP.cut.sum()))
for name, x in [("(a) pooled trials", EP.dur), ("(b) session means", sess)]:
    print(f"{name}: mean {x.mean():.3f} s, min {x.min():.3f}, max {x.max():.3f}; within tolerance:",
          [within(x.mean(), 2.39, 0.239), within(x.min(), 1.54, 0.154), within(x.max(), 4.05, 0.405)])
x = EP.dur[~EP.cut]
print(f"(a) without the cut trials: mean {x.mean():.3f}, min {x.min():.3f}, max {x.max():.3f}")
print(f"variant, pole_out_time - pole_in_time: mean {EP.table.mean():.3f}, min {EP.table.min():.3f}, max {EP.table.max():.3f}")

**Result: reproduced.** Over all trials pooled (reading a) the pole is in reach for 2.394 s on
average, 1.543 s at least and 4.049 s at most, the paper's 2.39 s and 1.54 to 4.05 s. Over
session means (reading b) the mean is 2.403 s but the range is 1.930 to 3.125 s, outside the
tolerance; the claim holds under reading (a). 617 of 5978 trials are cut by the end of the
recording, and the paper's range is reproduced with them in.

## behavior-whisking-bouts

> Mice whisked in bouts with a mean duration of 261 ms, a peak-to-peak amplitude of 15.7 deg
> and a frequency of 15.4 Hz, separated by rest.

A bout is a contiguous run of amplitude > 2.5 deg in one trial. Open details from the claims
file: (1) peak-to-peak as twice the bout's maximum Hilbert amplitude (`2amp`) or the largest
range of band-passed theta within a cycle of the bout (`range`); (2) whether touch breaks a
bout. Frequency is 1/cycle length over every whisking cycle (amplitude > 2.5 deg throughout),
pooled. As a variant not in the claims file, bouts holding at least one complete cycle.

In [ ]:
bout_rows = []
freqs = []
for c in cells:
    d = D[c.cell_no]
    wc = d["cyc"][d["cyc"].whisking]
    freqs.append(1000.0 / wc.dur.to_numpy())
    for tb in (False, True):
        bt = hires.bouts(c, d["T"], tb, d["cyc"])
        bout_rows.append(bt.assign(cell=c.cell_no, touch_breaks=tb))
BT = pd.concat(bout_rows, ignore_index=True)
freq = np.concatenate(freqs)
print(f"whisking cycles {len(freq)}: frequency mean {freq.mean():.2f} Hz, median {np.median(freq):.2f}; within 15.4 +- 1.54: {within(freq.mean(), 15.4, 1.54)}")
rows = []
for tb in (False, True):
    for only_cycles in (False, True):
        b = BT[(BT.touch_breaks == tb) & ((BT.n_cycles > 0) | (not only_cycles))]
        for p2p in ("2amp", "range"):
            amp = 2 * b.amp_max if p2p == "2amp" else b.p2p_cycle
            rows.append(dict(touch_breaks=tb, bouts=">=1 cycle (variant)" if only_cycles else "all runs", p2p=p2p, n_bouts=len(b),
                             dur_ms=b.dur.mean(), median_dur=b.dur.median(), amp_deg=amp.mean(),
                             dur_ok=within(b.dur.mean(), 261, 26.1), amp_ok=within(amp.mean(), 15.7, 1.57)))
BTS = pd.DataFrame(rows)
print(BTS.round(2).to_string(index=False))
print("bout duration range %d to %d ms; bouts of 1 to 10 ms: %.2f of all runs" % (BT.dur.min(), BT.dur.max(), np.mean(BT[~BT.touch_breaks].dur <= 10)))

**Result: partially reproduced.** The amplitude and the frequency match: twice the maximum
Hilbert amplitude averages 15.74 deg over bouts that touch does not break (15.7 in the paper)
and the whisking frequency 15.76 Hz (15.4). The mean bout duration does not: 91.6 ms with touch
not breaking bouts and 69.7 ms with it, against 261 ms, so no reading puts all three values in
tolerance. Most runs of amplitude above 2.5 deg are short; counting only bouts that hold a
complete cycle (not a reading of the claims file) gives 213.8 to 238.3 ms, but then the
amplitude is 33.9 to 35.7 deg.

## behavior-touches-reaction

> Mice touched the pole 2.33 times on average before responding, and the reaction time from
> first touch to first lick was 367 +- 234 ms (mean +- s.d.).

Reaction time: first lick (`first_lick_time`) minus the first touch onset, every trial with a
lick and a touch, pooled. Touches before the response: (a) trials with a lick, touches before
the first lick; (b) all trials with a touch, touches before the first lick or, without a lick,
touches in the sample epoch. Variants: only trials whose first lick follows the first touch,
and the first lick taken from the lick series.

In [ ]:
rt_rows = []
for c in cells:
    T = D[c.cell_no]["T"]
    on_t = c.seg_t0[T.trial] + (T.on - c.seg_start[T.trial]) / 1000
    for k in range(len(c.trials)):
        fl = c.trials.first_lick_time.iloc[k]
        ons = on_t[T.trial == k]
        in_pole = c.pole[T.on[T.trial == k]]
        a = c.seg_start[k]
        b = c.seg_start[k + 1] if k + 1 < len(c.seg_start) else len(c.vt)
        li = np.flatnonzero(c.lick[a:b])
        fl2 = c.seg_t0[k] + li[0] / 1000 if len(li) else np.nan
        rt_rows.append(dict(cell=c.cell_no, trial=k, lick=not np.isnan(fl), n_touch=len(ons),
                            rt=(fl - ons[0]) * 1000 if len(ons) and not np.isnan(fl) else np.nan,
                            rt_series=(fl2 - ons[0]) * 1000 if len(ons) and not np.isnan(fl2) else np.nan,
                            before_lick=int(np.sum(ons < fl)) if not np.isnan(fl) else np.nan,
                            in_sample=int(in_pole.sum())))
RT = pd.DataFrame(rt_rows)
rt = RT.rt.dropna()
print(f"reaction time, trials with a lick and a touch: n {len(rt)}, mean {rt.mean():.0f} ms, s.d. {rt.std():.0f}; negative (lick before first touch) {np.sum(rt < 0)}; within 367 +- 234: {within(rt.mean(), 367, 234)}")
rp = rt[rt > 0]
print(f"variant, first lick after the first touch: n {len(rp)}, mean {rp.mean():.0f} ms, s.d. {rp.std():.0f}")
rs = RT.rt_series.dropna()
print(f"variant, first lick from the lick series: n {len(rs)}, mean {rs.mean():.0f} ms, s.d. {rs.std():.0f}")
na = RT[RT.lick].before_lick
nb_ = np.where(RT.lick, RT.before_lick, RT.in_sample)[RT.n_touch > 0]
print(f"touches before the response: (a) {na.mean():.2f} over {len(na)} trials with a lick; (b) {np.mean(nb_):.2f} over {len(nb_)} trials with a touch; within 2.33 +- 0.233: {within(na.mean(), 2.33, 0.233)}, {within(np.mean(nb_), 2.33, 0.233)}")
nap = RT[RT.lick & (RT.rt > 0)].before_lick
print(f"variant, trials whose first lick follows the first touch: {nap.mean():.2f} touches over {len(nap)} trials")

**Result: partially reproduced.** The reaction time is 190 ms on average over 3093 trials with a
lick and a touch, inside the claim's tolerance only because the tolerance is the paper's s.d.
(367 +- 234 ms); its s.d. here is 700 ms, as 544 trials have their first lick before the first
touch. Counting only trials whose first lick follows the first touch (a variant) gives 420 +- 411
ms. The number of touches before the response is 3.00 under reading (a) and 3.51 under (b),
outside 2.33 +- 0.233 under both.

## example-cell-rates

> The example L4 C2 neuron of Figure 1 fired at 0.73 spikes/s in the 500 ms before the sample
> epoch and 1.91 spikes/s during it, a 2.6-fold increase (Wilcoxon rank sum, p = 3.38e-7).

For every L4 C2 cell: per-trial rates in the 500 ms before `pole_available` turns on and over
the time it is on; the cell's rates are the means of the per-trial rates (rates from pooled
spikes and time are shown too); the rank-sum test compares the per-trial rates.

In [ ]:
ex_rows = []
EP_by = {no: g for no, g in EP.groupby("cell")}
for no in G["L4C2"]:
    c = by_no[no]
    s = np.sort(D[no]["s"])
    e = EP_by[no]
    on_vt = e.trial.to_numpy() * hires.TRIAL_STRIDE_MS + e.on_ms.to_numpy()
    off_vt = e.trial.to_numpy() * hires.TRIAL_STRIDE_MS + e.off_ms.to_numpy()
    ok = e.on_ms.to_numpy() >= 500
    nb0 = hires.counts_in(s, on_vt[ok] - 500, 500)
    ns = np.searchsorted(s, off_vt[ok]) - np.searchsorted(s, on_vt[ok])
    dur = (off_vt[ok] - on_vt[ok]) / 1000
    rb, rs_ = nb0 / 0.5, ns / dur
    ex_rows.append(dict(cell=no, trials=int(ok.sum()), base=rb.mean(), samp=rs_.mean(), fold=rs_.mean() / rb.mean() if rb.mean() > 0 else np.inf,
                        base_pooled=nb0.sum() / (0.5 * ok.sum()), sample_pooled=ns.sum() / dur.sum(),
                        p=stats.ranksums(rb, rs_).pvalue))
EX = pd.DataFrame(ex_rows).set_index("cell")
EX["match"] = [within(r.base, 0.73, 0.073) and within(r.samp, 1.91, 0.191) and within(r.fold, 2.6, 0.26) and r.p < 0.05 for r in EX.itertuples()]
EX["distance"] = np.sqrt(((EX.base - 0.73) / 0.073) ** 2 + ((EX.samp - 1.91) / 0.191) ** 2 + ((EX.fold - 2.6) / 0.26) ** 2)
EX["fold_pooled"] = EX.sample_pooled / EX.base_pooled
print(EX.sort_values("distance").to_string(formatters={"p": "{:.3g}".format}, float_format="{:.3f}".format))
print("cell 17 has no trial whose pole comes into reach 500 ms or more after the start of the recording, so it has no baseline")
EXC = int(EX.distance.idxmin())
print("cells matching all three and p < 0.05:", EX.index[EX.match].tolist(), "| nearest:", EXC)

**Result: reproduced.** One L4 C2 cell matches, cell 23 (subject jf35892,
2010-02-19): 0.727 spikes/s in the 500 ms before the pole is in reach and 1.754 spikes/s during it
(mean of per-trial rates; 1.909 spikes/s from pooled spikes and time), 2.41-fold (2.63 pooled),
rank-sum p = 1.36e-7 over 121 trials. The paper gives 0.73, 1.91, 2.6-fold and p = 3.38e-7. The
next nearest cell is four times farther in the scaled distance (5.38 against 1.10). Cell 23 is the
example cell of the claims that follow.

## example-cell-sample-ff

> The Fano factor of spike counts over the whole sample epoch was large, 7.51, in the paragraph
> on the example neuron, which does not say whose value it is.

Per cell, the variance over the mean of the per-trial spike count while the pole is in reach.
Readings: (a) the cell of example-cell-rates; (b) the mean over the 41 L4 cells; (c) the mean
over all 52 cells.

In [ ]:
ff_sample = {}
for c in cells:
    s = np.sort(D[c.cell_no]["s"])
    e = EP_by[c.cell_no]
    on_vt = e.trial.to_numpy() * hires.TRIAL_STRIDE_MS + e.on_ms.to_numpy()
    off_vt = e.trial.to_numpy() * hires.TRIAL_STRIDE_MS + e.off_ms.to_numpy()
    ff_sample[c.cell_no] = hires.fano(np.searchsorted(s, off_vt) - np.searchsorted(s, on_vt))
ffs = pd.Series(ff_sample)
vals = {"(a) example cell %d" % EXC: ffs[EXC], "(b) mean over 41 L4 cells": ffs[G["L4C2"] + G["L4out"]].mean(),
        "(c) mean over 52 cells": ffs.mean()}
for k, v in vals.items():
    print(f"{k}: {v:.2f}; within 7.51 +- 0.751: {within(v, 7.51, 0.751)}")
print("L4 C2 cells within 7.51 +- 0.751:", ffs[G["L4C2"]][(ffs[G["L4C2"]] - 7.51).abs() <= 0.751].round(2).to_dict())

**Result: reproduced.** The sample-epoch Fano factor of cell 23 is 7.51
(reading a), the paper's value; the mean over all 52 cells is 7.89 (reading c, also within
tolerance) and over the 41 L4 cells 6.31 (reading b, outside it).

## example-cell-touch-window

> For the example neuron, 74.4% of spikes fell in a window 8 to 40 ms after touch onset, which
> covers 15.3% of exploration time.

As fig3c-touch-spikes-l4c2 (below), for the cell of example-cell-rates, under every reading of
the touch window. The window runs from the latency to latency + t_on.

In [ ]:
ew = W[(W.cell == EXC) & (W.lat_reading == "a")].copy()
ew["end_ms"] = ew.latency + ew.t_on
ew["ok_frac"] = [within(100 * v, 74.4, 7.44) for v in ew.frac_spikes]
ew["ok_start"] = [within(v, 8, 1) for v in ew.latency]
ew["ok_end"] = [within(v, 40, 4) for v in ew.end_ms]
ew["ok_time"] = [within(100 * v, 15.3, 1.53) for v in ew.frac_time]
ew["all_ok"] = ew[["ok_frac", "ok_start", "ok_end", "ok_time"]].all(axis=1)
print(ew[WKEY[1:] + ["latency", "t_on", "end_ms", "frac_spikes", "frac_time", "all_ok"]].round(3).to_string(index=False))
print("readings with all four within tolerance: %d of %d" % (ew.all_ok.sum(), len(ew)))
for st, g in ew.groupby("start"):
    print(f"start = {st}{' (the criterion)' if st == 'cross' else ' (variant)'}: spikes {100 * g.frac_spikes.min():.1f} to {100 * g.frac_spikes.max():.1f} %, "
          f"window {g.latency.min():.0f} to {g.end_ms.min():.0f}-{g.end_ms.max():.0f} ms, time {100 * g.frac_time.min():.1f} to {100 * g.frac_time.max():.1f} %, "
          f"all four within tolerance in {g.all_ok.sum()} of {len(g)}")

**Result: partially reproduced.** For cell 23 the transition cutoff and the literal one
give the same window under every reading: it starts at the 8 ms latency and ends at 36 to 38
ms (paper 8 to 40 ms, within tolerance). The fraction of exploration spikes in it is 78.4 to
79.1% when the closest onset is read on either side (within 74.4 +- 7.44%) and 83.6 to 84.6%
when it is read as the onset before the spike (outside). The window covers 19.0 to 20.2% of
exploration time under every reading, against 15.3 +- 1.53%. No reading puts all four values in
tolerance.

## text-spikes-per-touch

> Each touch evoked on the order of one spike (1.51 spikes per first touch, 0.31 per later
> touch) with an onset latency of 8 ms.

Readings: (a) the example cell, (b) the L4 C2 mean, (c) the L4 C2 median; spikes per touch in
the cell's touch window under every window reading; latency from first touches (and, as for
table1, from all touches).

In [ ]:
rows = []
wl = W[(W.lat_reading == "a")]
for key, g in wl[wl.cell.isin(G["L4C2"])].groupby(WKEY[1:]):
    g = g.set_index("cell")
    for lat_src in ("lat_first", "lat_all"):
        lat = base.loc[G["L4C2"], lat_src]
        for rd, f in [("(a) example", lambda x: x.loc[EXC]), ("(b) mean", np.nanmean), ("(c) median", np.nanmedian)]:
            v1, v2, v3 = f(g.spt_first), f(g.spt_later), f(lat)
            rows.append(dict(zip(WKEY[1:], key), latency_from=lat_src, reading=rd, first=v1, later=v2, latency=v3,
                             ok=within(v1, 1.51, 0.151) and within(v2, 0.31, 0.031) and within(v3, 8, 1)))
TS = pd.DataFrame(rows)
print(TS.groupby(["start", "reading", "latency_from"]).agg(first_min=("first", "min"), first_max=("first", "max"), later_min=("later", "min"),
                                                  later_max=("later", "max"), latency=("latency", "mean"), n_ok=("ok", "sum"), n=("ok", "size")).round(3))
print(TS[TS.ok].to_string(index=False) if TS.ok.any() else "no reading puts all three values in tolerance")

**Result: reproduced.** Under reading (a), the example cell, there are
1.474 to 1.505 spikes per first touch and 0.302 to 0.307 per later touch with an 8 ms latency
(9 ms from all touches), within tolerance under all 8 window readings of the transition cutoff
(and the 8 of the literal variant, which give the same window for this cell). The L4 C2 mean
(1.16 to 1.28 and 0.62 to 0.69 under the transition cutoff) and median (0.70 to 0.73 and 0.31
to 0.33), readings (b) and (c), do not match, which agrees with the claims file's note that the
sentence is about the example cell.

## text-whisking-rates

> Spike rates were low in both whisking (0.67 spikes/s) and non-whisking (0.23 spikes/s)
> periods.

Readings: (a) the example cell; (b) the L4 C2 mean; (c) the L4 C2 median; (d) all L4 C2 spikes
over all L4 C2 time pooled; whisking read without and with touch epochs.

In [ ]:
rows = []
tot = {}
for no in G["L4C2"]:
    c = by_no[no]
    T = D[no]["T"]
    s = D[no]["s"]
    si = hires.spike_samples(c, s)
    for wt in (False, True):
        m = hires.whisk_mask(c, T, wt)
        tot.setdefault(("w", wt), [0, 0])
        tot[("w", wt)][0] += m[si].sum()
        tot[("w", wt)][1] += m.sum() / 1000
    m = hires.nonwhisk_mask(c, T)
    tot.setdefault(("n",), [0, 0])
    tot[("n",)][0] += m[si].sum()
    tot[("n",)][1] += m.sum() / 1000
b4 = base.loc[G["L4C2"]]
for wt in (False, True):
    col = "rate_whisk_with_touch" if wt else "rate_whisk_no_touch"
    for rd, w_, n_ in [("(a) example", b4.loc[EXC, col], b4.loc[EXC, "rate_nonwhisk"]),
                       ("(b) mean", b4[col].mean(), b4.rate_nonwhisk.mean()),
                       ("(c) median", b4[col].median(), b4.rate_nonwhisk.median()),
                       ("(d) pooled", tot[("w", wt)][0] / tot[("w", wt)][1], tot[("n",)][0] / tot[("n",)][1])]:
        rows.append(dict(whisking_with_touch=wt, reading=rd, whisking=w_, nonwhisking=n_,
                         ok=within(w_, 0.67, 0.067) and within(n_, 0.23, 0.023)))
TW = pd.DataFrame(rows)
print(TW.round(3).to_string(index=False))
print("L4 C2 cells with both rates within tolerance (whisking without / with touch):",
      b4.index[[within(r.rate_whisk_no_touch, 0.67, 0.067) and within(r.rate_nonwhisk, 0.23, 0.023) for r in b4.itertuples()]].tolist(),
      b4.index[[within(r.rate_whisk_with_touch, 0.67, 0.067) and within(r.rate_nonwhisk, 0.23, 0.023) for r in b4.itertuples()]].tolist())

**Result: not reproduced.** No reading puts both rates in tolerance. The
example cell's non-whisking rate is 0.227 spikes/s (paper 0.23), but its whisking rate is 1.257
spikes/s with touch epochs left out and 2.373 with them in (paper 0.67). The L4 C2 means (2.40 or
3.98 and 1.30), medians (0.86 or 2.28 and 0.315) and pooled rates (2.63 or 3.97 and 1.41) are
further off.

## example-cell-phase-modulation

> During exploration the example neuron's spikes are coupled to whisking phase, with a
> modulation depth close to 1 and near zero spiking at phase 0.

In [ ]:
r = base.loc[EXC]
cen = np.linspace(-np.pi, np.pi, 13)[:-1] + np.pi / 12
print(f"cell {EXC}: modulation depth {r.depth_mod:.3f}, preferred phase {np.degrees(r.phi_pref):.0f} deg, from {int(r.phase_spikes)} spikes in {r.phase_time_s:.1f} s of phase-analysis cycles")
print("rate per 30 deg bin (spikes/s), bin centres in deg:")
print(pd.Series(D[EXC]["phase_rate"], index=np.round(np.degrees(cen)).astype(int)).round(2).to_string())
print("spike counts per bin:", D[EXC]["phase_cnt"].tolist())
print("within 1 +- 0.1:", within(r.depth_mod, 1, 0.1))
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.bar(np.degrees(cen), D[EXC]["phase_rate"], width=28, color="0.6")
ax.set_xlabel("whisking phase (deg)"); ax.set_ylabel("spikes/s"); ax.set_title(f"cell {EXC}")
plt.tight_layout(); plt.show()
print("plotted range: %.2f to %.2f spikes/s, none outside the axes" % (np.nanmin(D[EXC]["phase_rate"]), np.nanmax(D[EXC]["phase_rate"])))

**Result: reproduced.** Cell 23's modulation depth is 1.000, and
it fired no spike in the five 30 deg bins centred from -75 to 45 deg, around phase 0, as the
claim states. The fit rests on 13 spikes in 16.4 s of phase-analysis cycles, so the depth is not
precise.

## Table 1: table1-l4-c2, table1-l4-outside, table1-l5

> For the 31 L4 C2 cells Table 1 gives, as mean +- s.d. (median), spikes per first touch 1.36 +-
> 1.32 (0.87), per later touch 0.74 +- 0.87 (0.33), touch-evoked spikes 70.6 +- 20.9% (74.7%),
> touch and whisking spikes 75.2 +- 19.0% (80.3%), phase modulation depth 0.67 +- 0.32 (0.70),
> non-whisking rate 1.31 +- 2.88 (0.32) spikes/s, whisking rate 1.39 +- 2.33 (0.33) spikes/s,
> onset latency 7.8 +- 3.0 (8) ms and minimum ISI 2.9 +- 1.9 (2.3) ms.

(and the same measures for the 10 L4 cells outside C2 and the 11 L5 cells). Each value is
computed under every combination of readings: the touch window (8 with the transition cutoff,
the criterion's, and 8 more with the literal cutoff as a variant, and the two latency readings
for cells without a first-touch latency), the phase window width, whisking with or without
touch epochs, and the latency from first touches or from all touches. The criterion counts the
values within tolerance (15 of 18 for L4 C2 and L5, 13 of 17 for L4 outside C2).

Whisking periods are read here as the claims file defines them, outside touch epochs ("Touch
epochs are periods of contact. For other epochs theta is band-passed ... whisking epochs have
amplitude >2.5 deg"; the answer about fig4c-phase-decoding quotes the Methods to the same
effect). Whisking with touch epochs, which the whisking-rate claims list as a reading of their
own, is shown here as a variant.

In [ ]:
EXPECT = {
    "L4C2": [("spt_first", "mean", 1.36, 1.32), ("spt_first", "median", 0.87, 0.087), ("spt_later", "mean", 0.74, 0.87),
             ("spt_later", "median", 0.33, 0.033), ("touch_pct", "mean", 70.6, 20.9), ("touch_pct", "median", 74.7, 7.47),
             ("tp_pct", "mean", 75.2, 19.0), ("tp_pct", "median", 80.3, 8.03), ("depth", "mean", 0.67, 0.32),
             ("depth", "median", 0.70, 0.07), ("nonwhisk", "mean", 1.31, 2.88), ("nonwhisk", "median", 0.32, 0.032),
             ("whisk", "mean", 1.39, 2.33), ("whisk", "median", 0.33, 0.033), ("latency", "mean", 7.8, 3.0),
             ("latency", "median", 8, 1), ("min_isi", "mean", 2.9, 1.9), ("min_isi", "median", 2.3, 0.23)],
    "L4out": [("spt_first", "mean", 0.08, 0.19), ("spt_later", "mean", 0.02, 0.04), ("touch_pct", "mean", 9.2, 13.7),
              ("touch_pct", "median", 3.6, 0.36), ("tp_pct", "mean", 33.0, 30.2), ("tp_pct", "median", 23.8, 2.38),
              ("depth", "mean", 0.58, 0.36), ("depth", "median", 0.61, 0.061), ("nonwhisk", "mean", 2.20, 2.91),
              ("nonwhisk", "median", 1.10, 0.11), ("whisk", "mean", 2.41, 3.48), ("whisk", "median", 1.31, 0.131),
              ("latency", "mean", 18.5, 6.0), ("min_isi", "mean", 3.3, 1.3), ("min_isi", "median", 2.8, 0.28)],
    "L5": [("spt_first", "mean", 1.95, 2.96), ("spt_first", "median", 0.98, 0.098), ("spt_later", "mean", 1.28, 1.88),
           ("spt_later", "median", 0.42, 0.042), ("touch_pct", "mean", 24.0, 18.6), ("touch_pct", "median", 24.4, 2.44),
           ("tp_pct", "mean", 31.5, 16.4), ("tp_pct", "median", 36.0, 3.6), ("depth", "mean", 0.18, 0.12),
           ("depth", "median", 0.13, 0.013), ("nonwhisk", "mean", 12.0, 9.87), ("nonwhisk", "median", 12.8, 1.28),
           ("whisk", "mean", 16.2, 15.3), ("whisk", "median", 13.5, 1.35), ("latency", "mean", 9.7, 5.2),
           ("latency", "median", 8, 1), ("min_isi", "mean", 3.7, 2.5), ("min_isi", "median", 2.7, 0.27)],
}
# the L4 outside row of the claims file lists 17 values; its expected list above has the 15 of them that
# are numbers with a tolerance, and its two medians of spikes per touch (0 and 0) are checked as exact
EXPECT["L4out"] = EXPECT["L4out"][:2] + EXPECT["L4out"][2:]
THRESH = {"L4C2": (15, 10, 18), "L4out": (13, 9, 17), "L5": (15, 10, 18)}

per_cell = W.merge(PH, on=["cell", "group"] + WKEY, suffixes=("", "_ph"))
t1_rows = []
for g in GROUP_ORDER:
    sub = per_cell[per_cell.group == g]
    for key, kk in sub.groupby(PKEY):
        kk = kk.set_index("cell")
        for wt in (False, True):
            for lat_src in ("lat_first", "lat_all"):
                b = base.loc[kk.index]
                vals = dict(spt_first=kk.spt_first, spt_later=kk.spt_later, touch_pct=100 * kk.frac_spikes, tp_pct=100 * kk.frac_tp,
                            depth=b.depth_mod, nonwhisk=b.rate_nonwhisk,
                            whisk=b.rate_whisk_with_touch if wt else b.rate_whisk_no_touch, latency=b[lat_src], min_isi=b.min_isi_ms)
                row = dict(group=g, **dict(zip(PKEY, key)), whisk_with_touch=wt, latency_from=lat_src)
                n_ok = 0
                for m, stat, e, tol in EXPECT[g]:
                    v = msm(vals[m])[stat]
                    row[f"{m}_{stat}"] = v
                    ok = within(v, e, tol)
                    row[f"ok_{m}_{stat}"] = ok
                    n_ok += ok
                if g == "L4out":
                    row["spt_medians_zero"] = (np.nanmedian(vals["spt_first"]) == 0) and (np.nanmedian(vals["spt_later"]) == 0)
                row["n_ok"] = n_ok
                hi, lo, n = THRESH[g]
                row["verdict"] = "reproduced" if n_ok >= hi else ("partially" if n_ok >= lo else "not reproduced")
                t1_rows.append(row)
T1 = pd.DataFrame(t1_rows)
# the criterion's readings: the transition cutoff (answer in questions/000013.yaml) and whisking periods outside touch
# epochs (the claims file's definitions, "For other epochs ... whisking epochs have amplitude >2.5 deg")
T1["criterion"] = (T1.start == "cross") & ~T1.whisk_with_touch
FIRST = dict(lat_reading="a", shuffle="uniform", nearest="before", tmax=100, start="cross", width="pm", whisk_with_touch=False, latency_from="lat_first")


def first_reading(t):
    """The values under the first reading of every open detail, in the claims file's order."""
    r = t[np.logical_and.reduce([t[k] == v for k, v in FIRST.items()])].iloc[0]
    return [(f"{m}_{stat}", round(float(r[f"{m}_{stat}"]), 3), e, tol, bool(r[f"ok_{m}_{stat}"])) for m, stat, e, tol in EXPECT[r["group"]]]
for g in GROUP_ORDER:
    t = T1[T1.group == g]
    print(f"\n{g}: {len(t)} reading combinations; values within tolerance: min {t.n_ok.min()}, max {t.n_ok.max()} (of {len(EXPECT[g])} numeric values)")
    print(t.verdict.value_counts().to_string())
    print(t.groupby("start").n_ok.agg(["min", "max"]).to_string())
    cols = [c for c in t.columns if c.startswith("ok_")]
    print("fraction of combinations in which each value is within tolerance:")
    print(t[cols].mean().round(2).to_string())
    num = [c for c in t.columns if c.endswith("_mean") or c.endswith("_median")]
    print("range of each value over the combinations:")
    print(pd.DataFrame({"min": t[num].min(), "max": t[num].max()}).round(3).to_string())
    if g == "L4out":
        print("medians of spikes per first and later touch both 0:", t.spt_medians_zero.value_counts().to_dict())

In [ ]:
t = T1[T1.group == "L4C2"]
print("L4 C2: verdicts over the reading combinations:", t.verdict.value_counts().to_dict())
print("medians within tolerance, per combination: min %d, max %d of 9" % (t[[c for c in t.columns if c.startswith("ok_") and c.endswith("median")]].sum(axis=1).min(),
                                                                         t[[c for c in t.columns if c.startswith("ok_") and c.endswith("median")]].sum(axis=1).max()))
print(t.groupby(["start", "tmax"]).n_ok.agg(["min", "max"]))
print("verdicts by start of the cutoff search (cross is the criterion's, first a variant) and whisking reading:")
print(t.groupby(["start", "whisk_with_touch", "latency_from"]).n_ok.agg(["min", "max"]).to_string())
print(t.groupby("start").verdict.value_counts().unstack(fill_value=0).to_string())
c = t[t.criterion]
print("under the criterion's readings (cross, whisking outside touch epochs): values within tolerance %d to %d over %d combinations;" % (c.n_ok.min(), c.n_ok.max(), len(c)),
      "verdicts", c.verdict.value_counts().to_dict())
print(c.groupby(["tmax", "width"]).n_ok.agg(["min", "max"]).to_string())
print("values under the first reading of each open detail (measured, expected, tolerance, within):")
for row in first_reading(t):
    print("  ", row)

**Result: reproduced.** Under the criterion's readings, the transition
cutoff with whisking periods outside touch epochs, 15 of the 18 values are within tolerance in
all 64 combinations of the other readings (the 100 and 200 ms curves, the shuffle, the closest
onset, the phase window width and the latency PCTH), at the threshold for reproduced. The three
outside tolerance are medians: spikes per first touch (0.704 under the first reading of each
detail, against 0.87 +- 0.087), modulation depth (1.0 against 0.70) and whisking rate (0.86
against 0.33). Spikes evoked by touch are 73.8% (median 80.0%) and by touch or phase 80.9%
(82.7%) under the first reading. With touch epochs in the whisking rate (a variant) the mean
whisking rate also falls outside and the count is 14 (partially); with the literal cutoff (a
variant) it is 9 to 14. Fourteen of the 25 cells with a phase fit have a depth of 1; the median
L4 C2 cell has 4 spikes in the phase-analysis cycles.

In [ ]:
t = T1[T1.group == "L4out"]
print("L4 outside C2: verdicts over the reading combinations:", t.verdict.value_counts().to_dict())
print(t.groupby(["start", "lat_reading"]).n_ok.agg(["min", "max"]))
print("verdicts by start of the cutoff search (cross is the criterion's, first a variant) and whisking reading:")
print(t.groupby(["start", "whisk_with_touch", "latency_from"]).n_ok.agg(["min", "max"]).to_string())
print(t.groupby("start").verdict.value_counts().unstack(fill_value=0).to_string())
c = t[t.criterion]
print("under the criterion's readings (cross, whisking outside touch epochs): values within tolerance %d to %d over %d combinations;" % (c.n_ok.min(), c.n_ok.max(), len(c)),
      "verdicts", c.verdict.value_counts().to_dict())
print(c.groupby(["tmax", "width"]).n_ok.agg(["min", "max"]).to_string())
print("values under the first reading of each open detail (measured, expected, tolerance, within):")
for row in first_reading(t):
    print("  ", row)

**Result: blocked (methods_underspecified).** Under the criterion's
readings the count of values within tolerance is 7 to 9 of 17, which falls on both sides of the
threshold for partially reproduced (9): 52 combinations give not reproduced and 12 partially,
the difference coming from the touch-window readings. The medians of spikes per touch are not 0
under any reading; the means of spikes per touch (0.31 and 0.15 under the first reading,
against 0.08 and 0.02), the touch-evoked fraction (26.2%, median 23.8%, against 9.2% and 3.6%)
and the latency mean (8.1 ms over the cells with a latency, against 18.5) are outside
tolerance. Rates, mean depth and the minimum ISI are within it. The literal cutoff, a variant,
gives 10 to 11.

In [ ]:
t = T1[T1.group == "L5"]
print("L5: verdicts over the reading combinations:", t.verdict.value_counts().to_dict())
print(t.groupby(["start", "lat_reading"]).n_ok.agg(["min", "max"]))
print("verdicts by start of the cutoff search (cross is the criterion's, first a variant) and whisking reading:")
print(t.groupby(["start", "whisk_with_touch", "latency_from"]).n_ok.agg(["min", "max"]).to_string())
print(t.groupby("start").verdict.value_counts().unstack(fill_value=0).to_string())
c = t[t.criterion]
print("under the criterion's readings (cross, whisking outside touch epochs): values within tolerance %d to %d over %d combinations;" % (c.n_ok.min(), c.n_ok.max(), len(c)),
      "verdicts", c.verdict.value_counts().to_dict())
print(c.groupby(["tmax", "width"]).n_ok.agg(["min", "max"]).to_string())
print("values under the first reading of each open detail (measured, expected, tolerance, within):")
for row in first_reading(t):
    print("  ", row)

**Result: partially reproduced.** Under the criterion's readings 11 to
13 of the 18 values are within tolerance in every combination, partially reproduced throughout.
Rates, mean depth and median depth (0.226 and 0.131), the means of spikes per touch and the
touch-evoked fraction (27.2%, median 22.1% under the first reading) are within tolerance. The
medians of spikes per first and later touch (0.227 and 0.188 against 0.98 and 0.42), the median
whisking rate (11.0 against 13.5), the median latency (4 against 8 ms) and the median minimum
ISI (1.5 against 2.7 ms) are not. The literal cutoff, a variant, gives 9 to 11.

## latency-determinable

> A touch-response latency could be determined for all 31 L4 C2 cells but for only four of the
> 10 L4 cells outside C2, for lack of a touch response.

Readings of the PCTH: first touches or all touches. The search runs 0 to 50 ms.

In [ ]:
for src in ("lat_first", "lat_all"):
    has = base[src].notna().groupby(base.group).sum()
    print(f"{src}: cells with a latency {has.to_dict()}; within (31, 4): {has['L4C2'] == 31 and has['L4out'] == 4}")
print(base.loc[G["L4out"] + G["L5"], ["group", "lat_first", "lat_all"]].to_string())

**Result: not reproduced.** A latency is found for all 31 L4 C2 cells
under both readings, but for 7 of the 10 L4 cells outside C2 from first touches and 10 of 10
from all touches, not 4. The two readings agree that the count is not the paper's.

## l4-outside-weaker-touch

> L4 neurons outside C2 respond much more weakly to touch than L4 neurons in C2, for first and
> for later touches.

Readings of the measure: (a) spikes per touch in the touch window (every window reading), or
(b) the peak touch rate, the highest 1 ms bin of the first-touch PCTH and of the later-touch
PCTH, 0 to 50 ms after onset.

In [ ]:
rows = []
for key, g in W[W.group.isin(["L4C2", "L4out"])].groupby(WKEY):
    a, b = g[g.group == "L4C2"], g[g.group == "L4out"]
    pf, pl = ranksum(a.spt_first, b.spt_first), ranksum(a.spt_later, b.spt_later)
    rows.append(dict(zip(WKEY, key), med_c2_first=a.spt_first.median(), med_out_first=b.spt_first.median(), p_first=pf,
                     med_c2_later=a.spt_later.median(), med_out_later=b.spt_later.median(), p_later=pl,
                     both=(pf < 0.05) and (pl < 0.05) and a.spt_first.median() > b.spt_first.median() and a.spt_later.median() > b.spt_later.median()))
LW = pd.DataFrame(rows)
print("(a) spikes per touch: combinations with both tests p < 0.05 and C2 higher: %d of %d" % (LW.both.sum(), len(LW)))
print(LW.groupby("start").both.agg(["sum", "size"]).to_string())
print(LW.groupby("start")[["p_first", "p_later"]].agg(["min", "max"]).to_string(float_format="{:.3g}".format))
peak = {}
for no in G["L4C2"] + G["L4out"]:
    c = by_no[no]
    T = D[no]["T"]
    s = D[no]["s"]
    pk = []
    for sel in (T.order == 1, T.order > 1):
        t_, h, n = hires.pcth(c, s, T.on[sel])
        pk.append(h[(t_ >= 0)].max() * 1000)
    peak[no] = pk
peak = pd.DataFrame(peak, index=["peak_first", "peak_later"]).T
pa, pb = peak.loc[G["L4C2"]], peak.loc[G["L4out"]]
print("(b) peak touch rate (spikes/s): first touches C2 median %.1f, outside %.1f, p = %.3g; later touches C2 %.1f, outside %.1f, p = %.3g" %
      (pa.peak_first.median(), pb.peak_first.median(), ranksum(pa.peak_first, pb.peak_first),
       pa.peak_later.median(), pb.peak_later.median(), ranksum(pa.peak_later, pb.peak_later)))

**Result: reproduced.** Under reading (a) both rank-sum tests give p
< 0.05 with the C2 median higher in all 16 window readings of the transition cutoff (first
touches p = 1.0e-4 to 0.0043, later touches 2.0e-4 to 0.015; the paper gives 1.4e-5 and
6.32e-6), and in the 16 of the literal variant. Under reading (b) the peak touch rate is 148.1
against 25.0 spikes/s for first touches (p = 1.63e-5) and 51.3 against 5.6 for later touches (p
= 1.07e-5).

## l5-higher-rates

> L5 neurons near C2 have much higher spike rates than L4 neurons, with more diverse touch
> responses.

In [ ]:
for wt, col in ((False, "rate_whisk_no_touch"), (True, "rate_whisk_with_touch")):
    print(f"whisking read {'with' if wt else 'without'} touch epochs")
    allok, c2ok = True, True
    for rate in ("rate_nonwhisk", col):
        m = {g: base.loc[G[g], rate].median() for g in GROUP_ORDER}
        for other in ("L4C2", "L4out"):
            p = ranksum(base.loc[G["L5"], rate], base.loc[G[other], rate])
            ok = (m["L5"] > m[other]) and p < 0.05
            allok &= ok
            if other == "L4C2":
                c2ok &= ok
            print(f"  {rate}: L5 median {m['L5']:.2f} vs {other} {m[other]:.2f} spikes/s, rank sum p = {p:.3g}")
    print("  all four comparisons hold:", allok, "| those against L4 C2 hold:", c2ok)
fig, axs = plt.subplots(1, 3, figsize=(10, 2.6), sharey=False)
for ax, g in zip(axs, GROUP_ORDER):
    for no in G[g]:
        T = D[no]["T"]
        t_, h, n = hires.pcth(by_no[no], D[no]["s"], T.on[T.order == 1])
        ax.plot(t_, h * 1000, lw=0.7)
    ax.set_title(g); ax.set_xlabel("ms from first touch")
axs[0].set_ylabel("spikes/s")
plt.tight_layout(); plt.show()
print("plotted: first-touch PCTHs, 1 ms bins, -50 to 50 ms; all values inside the axes")

**Result: reproduced.** L5 medians are 12.90 spikes/s outside whisking and
10.96 (11.33 with touch epochs) during it, against 0.32 and 0.86 (2.28) for L4 C2 and 1.09 and
1.75 (1.78) for L4 outside C2; all four rank-sum tests give p < 0.05 under both readings (largest
p = 0.0092). The diversity of touch responses is not judged; the first-touch PCTHs are plotted.

## phase-mod-l4-inside-outside and phase-mod-l4-vs-l5

> Modulation by whisking phase does not differ between L4 neurons inside and outside C2.

> L4 neurons, inside and outside C2, are more modulated by whisking phase than L5 neurons.

Rank-sum tests of the per-cell modulation depth; cells without spikes in the phase-analysis
cycles have no depth and are left out (their numbers are printed).

In [ ]:
dm = {g: base.loc[G[g], "depth_mod"] for g in GROUP_ORDER}
print({g: f"n {v.notna().sum()} of {len(v)}, median {v.median():.3f}, mean {v.mean():.3f}" for g, v in dm.items()})
p = ranksum(dm["L4C2"], dm["L4out"])
print(f"L4 C2 vs L4 outside: rank sum p = {p:.3g}; absence holds at 0.05: {p >= 0.05}")
print("cells with depth 1 (no fitted baseline):", {g: int((v > 0.999).sum()) for g, v in dm.items()})
print("phase-analysis spikes per cell, median:", {g: base.loc[G[g], "phase_spikes"].median() for g in GROUP_ORDER})

**Result: reproduced.** The rank-sum test of modulation depth
between L4 C2 (25 cells with a fit, median 1.000) and L4 outside C2 (10 cells, median 0.994)
gives p = 0.942, so the paper's test finds no difference (paper p = 0.68). Both medians are
inflated by cells with few phase spikes (median 4 and 18 per cell) whose fit has no baseline.

In [ ]:
for g in ("L4C2", "L4out"):
    p = ranksum(dm[g], dm["L5"])
    print(f"{g} vs L5: medians {dm[g].median():.3f} vs {dm['L5'].median():.3f}, rank sum p = {p:.3g}, holds: {p < 0.05 and dm[g].median() > dm['L5'].median()}")
# variant: only cells with at least 20 phase-analysis spikes
sel = base.phase_spikes >= 20
print("variant, cells with >= 20 phase spikes:", {g: int(sel[G[g]].sum()) for g in GROUP_ORDER})
for a_, b_ in (("L4C2", "L4out"), ("L4C2", "L5"), ("L4out", "L5")):
    x, y = dm[a_][sel[G[a_]]], dm[b_][sel[G[b_]]]
    print(f"  {a_} {x.median():.3f} vs {b_} {y.median():.3f}: p = {ranksum(x, y):.3g}")

**Result: reproduced.** L4 C2 (median 1.000) and L4 outside C2 (0.994)
are both more phase modulated than L5 (0.131): p = 2.55e-4 and 0.0039 (paper 4.3e-5 and 0.012).
With only cells that have at least 20 phase spikes (6, 5 and 10 cells), the p values are 0.051 and
0.050.

## whisking-rate-l4-no-change and whisking-rate-l5-increase

> Across L4 neurons, inside and outside C2, spike rate does not change between whisking and
> non-whisking periods.

> L5 neurons fire modestly but significantly more during whisking than during non-whisking
> periods.

Wilcoxon signed-rank test of per-cell whisking against non-whisking rate, each group apart,
whisking read with and without touch epochs.

In [ ]:
WR = []
for g in GROUP_ORDER:
    for wt, col in ((False, "rate_whisk_no_touch"), (True, "rate_whisk_with_touch")):
        b = base.loc[G[g]]
        diff = b[col] - b.rate_nonwhisk
        p = stats.wilcoxon(b[col], b.rate_nonwhisk).pvalue
        WR.append(dict(group=g, whisking_with_touch=wt, median_whisk=b[col].median(), median_nonwhisk=b.rate_nonwhisk.median(),
                       median_difference=diff.median(), n_higher=int((diff > 0).sum()), n=len(b), p=p))
WR = pd.DataFrame(WR)
print(WR.to_string(index=False, float_format="{:.3g}".format))

**Result: partially reproduced.** For L4 outside C2 the
signed-rank test finds no change (p = 0.432 under both readings, paper 0.92). For L4 C2 it finds
one: 26 of 31 cells fire more when whisking with touch epochs left out (median difference +0.63
spikes/s, p = 5.0e-5) and 30 of 31 with them in (+1.21 spikes/s, p = 2.8e-9), where the paper
gives p = 0.75.

In [ ]:
print(WR[WR.group == "L5"].to_string(index=False, float_format="{:.3g}".format))
for r in WR[WR.group == "L5"].itertuples():
    print(f"whisking with touch epochs {r.whisking_with_touch}: p < 0.05 {r.p < 0.05}, median of the per-cell differences positive {r.median_difference > 0}")

**Result: reproduced.** The signed-rank test is
significant under both readings of whisking periods (p = 0.024 without touch epochs and 0.014
with them, paper 0.019), and the median of the per-cell differences, the measure the answered
criterion names, is positive under both (+1.88 and +1.79 spikes/s), with 9 and 10 of the 11
cells firing more when whisking. The group medians, shown only for comparison, run the other
way (11.0 and 11.3 spikes/s when whisking against 12.9 outside whisking; Table 1 gives 13.5 and
12.8).

## Figure 3C: fig3c-touch-spikes-l4c2, fig3c-touch-spikes-outside, fig3c-touch-spikes-l5

> In L4 C2 neurons 70.6 +- 20.9% (mean +- s.d.) of spikes during exploration are coupled to
> touch, within an average of 14.9% of exploration time.

> L4 neurons outside C2 have far fewer touch-coupled spikes, 8.4 +- 13.3%, in 3.5% of
> exploration time.

> L5 neurons have fewer touch-coupled spikes than L4 C2 neurons, 24.0 +- 18.6%, in 19.0% of
> exploration time.

Every touch-window reading is shown. For the outside and L5 groups, the rank-sum test against
L4 C2 uses the same reading for both groups; the variant sets cells without a first-touch
latency to 0%.

In [ ]:
F3 = []
c2 = W[W.group == "L4C2"]
for g, exp in (("L4C2", (70.6, 20.9, 14.9, 1.49)), ("L4out", (8.4, 13.3, 3.5, 0.35)), ("L5", (24.0, 18.6, 19.0, 1.9))):
    for key, kk in W[W.group == g].groupby(WKEY):
        ref = c2[(c2.lat_reading == "a") & (c2.shuffle == key[1]) & (c2.nearest == key[2]) & (c2.tmax == key[3]) & (c2.start == key[4])]
        fs = 100 * kk.frac_spikes
        nolat = base.loc[kk.cell, "lat_first"].isna().to_numpy()
        fs0 = np.where(nolat, 0.0, fs)
        p = ranksum(ref.frac_spikes, kk.frac_spikes) if g != "L4C2" else np.nan
        F3.append(dict(group=g, **dict(zip(WKEY, key)), touch_mean=fs.mean(), touch_sd=fs.std(ddof=1), touch_median=fs.median(),
                       time_mean=100 * kk.frac_time.mean(), p_vs_c2=p, lower_than_c2=fs.median() < 100 * ref.frac_spikes.median(),
                       ok_values=within(fs.mean(), exp[0], exp[1]) and within(100 * kk.frac_time.mean(), exp[2], exp[3]),
                       variant_zero_mean=fs0.mean()))
F3 = pd.DataFrame(F3)
F3["ok"] = F3.ok_values & ((F3.group == "L4C2") | ((F3.p_vs_c2 < 0.05) & F3.lower_than_c2))

In [ ]:
t = F3[(F3.group == "L4C2") & (F3.lat_reading == "a")]
print(t.drop(columns=["group", "lat_reading", "p_vs_c2", "lower_than_c2", "variant_zero_mean"]).round(2).to_string(index=False))
print("readings within both tolerances: %d of %d" % (t.ok.sum(), len(t)))
print("by start and longest window (cross is the criterion's start, first a variant):")
print(t.groupby(["start", "tmax"]).agg(ok=("ok", "sum"), n=("ok", "size"), touch_min=("touch_mean", "min"), touch_max=("touch_mean", "max"),
                                       time_min=("time_mean", "min"), time_max=("time_mean", "max")).round(2).to_string())
fig, ax = plt.subplots(figsize=(4, 3))
k = W[(W.group == "L4C2") & (W.lat_reading == "a") & (W.shuffle == "uniform") & (W.nearest == "before") & (W.tmax == 100) & (W.start == "cross")]
for g, col in (("L4C2", "red"), ("L4out", "grey"), ("L5", "orange")):
    kk = W[(W.group == g) & (W.lat_reading == "a") & (W.shuffle == "uniform") & (W.nearest == "before") & (W.tmax == 100) & (W.start == "cross")]
    ax.scatter(100 * kk.frac_time, 100 * kk.frac_spikes, s=12, c=col, label=g)
ax.set_xlabel("% of exploration time"); ax.set_ylabel("% of spikes in touch window"); ax.legend(fontsize=7)
ax.set_title("uniform, before, 100 ms, cross", fontsize=8)
plt.tight_layout(); plt.show()
kk = W[(W.lat_reading == "a") & (W.shuffle == "uniform") & (W.nearest == "before") & (W.tmax == 100) & (W.start == "cross")]
print("plotted ranges: time %.1f to %.1f %%, spikes %.1f to %.1f %%" % (100 * kk.frac_time.min(), 100 * kk.frac_time.max(), 100 * kk.frac_spikes.min(), 100 * kk.frac_spikes.max()))

**Result: blocked (methods_underspecified).** Under the transition
cutoff, the criterion's, 66.9 to 75.8% of exploration spikes fall in touch windows covering 14.6
to 17.6% of exploration time. All 4 readings with the 100 ms curve put both values in tolerance
(70.6 +- 20.9% and 14.9 +- 1.49%), but only 1 of the 4 with the 200 ms curve does: the other
three cover 16.5 to 17.6% of exploration time. The readings of the curve's range, which the
paper does not give, disagree. The literal cutoff, a variant, gives 38.6 to 64.5% in 8.5 to
13.4% of time.

In [ ]:
t = F3[F3.group == "L4out"]
print(t.drop(columns=["group"]).round(3).to_string(index=False, formatters={"p_vs_c2": "{:.3g}".format}))
print("readings meeting the criterion: %d of %d" % (t.ok.sum(), len(t)))
print(t.groupby(["start", "lat_reading", "tmax"]).agg(ok=("ok", "sum"), n=("ok", "size"), touch_min=("touch_mean", "min"), touch_max=("touch_mean", "max"),
                                                      time_min=("time_mean", "min"), time_max=("time_mean", "max"), p_max=("p_vs_c2", "max")).round(3).to_string())

**Result: blocked (methods_underspecified).** The claim depends
on fig3c-touch-spikes-l4c2, which is blocked. Under every one of its 16 readings with the
transition cutoff the time fraction is outside tolerance: 18.8 to 31.7% of spikes in 17.5 to
26.3% of exploration time (paper 8.4 +- 13.3% in 3.5%), with the rank-sum test against L4 C2 at
p < 0.001. The literal cutoff, a variant, gives 1.5 to 7.4% in 1.2 to 2.6%.

In [ ]:
t = F3[F3.group == "L5"]
print(t.drop(columns=["group"]).round(3).to_string(index=False, formatters={"p_vs_c2": "{:.3g}".format}))
print("readings meeting the criterion: %d of %d" % (t.ok.sum(), len(t)))
print(t.groupby(["start", "lat_reading", "tmax"]).agg(ok=("ok", "sum"), n=("ok", "size"), touch_min=("touch_mean", "min"), touch_max=("touch_mean", "max"),
                                                      time_min=("time_mean", "min"), time_max=("time_mean", "max"), p_max=("p_vs_c2", "max")).round(3).to_string())

**Result: blocked (methods_underspecified).** The claim depends on
fig3c-touch-spikes-l4c2, which is blocked. Under the transition cutoff 4 of the 8 window
readings meet the criterion under both latency readings (20.8 to 30.6% of spikes in 15.4 to
22.5% of time across the 8, paper 24.0 +- 18.6% in 19.0%; p against L4 C2 below 0.001), 2 with
the 100 ms curve and 2 with the 200 ms curve. The literal cutoff, a variant, gives 5.1 to 14.1%
in 3.2 to 8.2% of time.

## Figure 3D: fig3d-touch-phase-l4c2, fig3d-touch-phase-outside, fig3d-touch-phase-l5, fig3d-phase-only

> In L4 C2 neurons 75.2 +- 19.0% of exploration spikes are coupled to touch or to whisking
> phase, within an average of 22.0% of exploration time.

> L4 neurons outside C2 have fewer touch or phase coupled spikes than L4 C2, 33.0 +- 30.2% in
> 16.9% of exploration time.

> L5 neurons have fewer touch or phase coupled spikes than L4 C2, 31.5 +- 16.4% in 26.1% of
> exploration time.

> Spikes coupled to whisking phase alone are 4.6 +- 8.7% in L4 C2, 23.8 +- 27.9% in L4 outside
> C2 and 7.6 +- 14.9% in L5.

In [ ]:
F3D = []
c2p = PH[PH.group == "L4C2"]
for g, exp in (("L4C2", (75.2, 19.0, 22.0, 2.2)), ("L4out", (33.0, 30.2, 16.9, 1.69)), ("L5", (31.5, 16.4, 26.1, 2.61))):
    for key, kk in PH[PH.group == g].groupby(PKEY):
        kd = dict(zip(PKEY, key))
        ref = c2p[(c2p.lat_reading == "a") & (c2p.shuffle == kd["shuffle"]) & (c2p.nearest == kd["nearest"]) & (c2p.tmax == kd["tmax"]) &
                  (c2p.start == kd["start"]) & (c2p.width == kd["width"])]
        tp = 100 * kk.frac_tp
        p = ranksum(ref.frac_tp, kk.frac_tp) if g != "L4C2" else np.nan
        F3D.append(dict(group=g, **kd, tp_mean=tp.mean(), tp_sd=tp.std(ddof=1), time_mean=100 * kk.time_tp.mean(),
                        phase_only_mean=100 * kk.frac_phase_only.mean(), phase_only_sd=100 * kk.frac_phase_only.std(ddof=1),
                        p_vs_c2=p, lower_than_c2=tp.median() < 100 * ref.frac_tp.median(),
                        ok_values=within(tp.mean(), exp[0], exp[1]) and within(100 * kk.time_tp.mean(), exp[2], exp[3])))
F3D = pd.DataFrame(F3D)
F3D["ok"] = F3D.ok_values & ((F3D.group == "L4C2") | ((F3D.p_vs_c2 < 0.05) & F3D.lower_than_c2))

In [ ]:
t = F3D[(F3D.group == "L4C2") & (F3D.lat_reading == "a")]
print(t.drop(columns=["group", "lat_reading", "p_vs_c2", "lower_than_c2"]).round(2).to_string(index=False))
print("readings within both tolerances: %d of %d" % (t.ok.sum(), len(t)))
print(t.groupby(["start", "lat_reading", "tmax", "width"]).agg(ok=("ok", "sum"), n=("ok", "size"), tp_min=("tp_mean", "min"), tp_max=("tp_mean", "max"),
                                                               time_min=("time_mean", "min"), time_max=("time_mean", "max")).round(2).to_string())

**Result: blocked (methods_underspecified).** The claim depends on
fig3c-touch-spikes-l4c2. Under the transition cutoff 74.6 to 81.7% of spikes are touch or phase
coupled in 22.3 to 28.1% of exploration time, and 4 of the 16 readings put both values in
tolerance (75.2 +- 19.0% in 22.0 +- 2.2%), all with the 100 ms curve and the phase window t_w
wide in total; the others cover 24.9 to 28.1% of time. The literal cutoff, a variant, gives
44.3 to 68.1% in 11.7 to 18.4%.

In [ ]:
t = F3D[F3D.group == "L4out"]
print(t.groupby(["start", "lat_reading", "width"])[["tp_mean", "time_mean", "p_vs_c2"]].agg(["min", "max"]).to_string(float_format="{:.3g}".format))
print("readings meeting the criterion: %d of %d" % (t.ok.sum(), len(t)))
print(t.groupby(["start", "lat_reading", "tmax", "width"]).agg(ok=("ok", "sum"), n=("ok", "size"), tp_min=("tp_mean", "min"), tp_max=("tp_mean", "max"),
                                                               time_min=("time_mean", "min"), time_max=("time_mean", "max")).round(2).to_string())

**Result: blocked (methods_underspecified).** The claim depends on
fig3d-touch-phase-l4c2. None of the 32 readings with the transition cutoff meets the criterion:
the touch or phase fraction is 40.6 to 60.2% (within 33.0 +- 30.2%), but the time fraction is 27.0
to 42.0% against 16.9 +- 1.69%. One reading of the literal variant meets it (34.3% in 16.0%).

In [ ]:
t = F3D[F3D.group == "L5"]
print(t.groupby(["start", "lat_reading", "width"])[["tp_mean", "time_mean", "p_vs_c2"]].agg(["min", "max"]).to_string(float_format="{:.3g}".format))
print("readings meeting the criterion: %d of %d" % (t.ok.sum(), len(t)))
print(t.groupby(["start", "lat_reading", "tmax", "width"]).agg(ok=("ok", "sum"), n=("ok", "size"), tp_min=("tp_mean", "min"), tp_max=("tp_mean", "max"),
                                                               time_min=("time_mean", "min"), time_max=("time_mean", "max")).round(2).to_string())

**Result: blocked (methods_underspecified).** The claim depends on
fig3d-touch-phase-l4c2. Under the transition cutoff 8 of 32 readings meet the criterion, all
with the 100 ms curve and the phase window t_w wide in total (36.4 to 37.5% in 26.3 to 28.1% of
time, paper 31.5 +- 16.4% in 26.1 +- 2.61%); over all 32 the fraction is 36.4 to 52.7% in 26.3
to 41.0% of time. None of the literal variant does (6.1 to 24.7%).

In [ ]:
po = F3D.pivot_table(index=PKEY, columns="group", values="phase_only_mean")
po = po.reset_index()
po = po[po.lat_reading == "a"].drop(columns="lat_reading").merge(
    F3D[(F3D.lat_reading == "b")].pivot_table(index=PKEY[1:], columns="group", values="phase_only_mean")[["L4out", "L5"]].add_suffix("_latb").reset_index(),
    on=PKEY[1:])
po["ok"] = [within(r.L4C2, 4.6, 8.7) and within(r.L4out, 23.8, 27.9) and within(r.L5, 7.6, 14.9) and r.L4out > max(r.L4C2, r.L5) for r in po.itertuples()]
po["ok_latb"] = [within(r.L4C2, 4.6, 8.7) and within(r.L4out_latb, 23.8, 27.9) and within(r.L5_latb, 7.6, 14.9) and r.L4out_latb > max(r.L4C2, r.L5_latb) for r in po.itertuples()]
print(po.round(2).to_string(index=False))
print("readings meeting the criterion: latency reading (a) %d, (b) %d, of %d" % (po.ok.sum(), po.ok_latb.sum(), len(po)))
print(po.groupby("start")[["ok", "ok_latb"]].agg(["sum", "size"]).to_string())

**Result: blocked (methods_underspecified).** The claim depends on
fig3d-touch-phase-l4c2. Under the transition cutoff the phase-only fractions meet the criterion
in 11 of 16 readings (L4 C2 5.1 to 9.3%, L4 outside 20.3 to 32.1%, L5 10.3 to 25.4%); in the 5
that fail the L5 mean is 23.3 to 25.4%, above its tolerance (7.6 +- 14.9%). All 16 of the
literal variant meet it.

## Figure 4A: fig4a-touch-detection-c2 and fig4a-touch-detection-outside

> Pooling the activity of only 15 L4 C2 neurons, integrated over 10 ms, is enough to detect
> 95% of touches.

> Touch detection by neurons from surrounding barrels, outside C2, is poor.

Touch-aligned responses from -30 to +50 ms, leaving out touches with another onset within 50
ms; N neurons drawn with replacement, 1000 responses each with replacement, 10 ms causal
integration, pooled; one threshold per post-touch time point, chosen on half the samples to
maximise percent correct, tested on the other half. Open details from the claims file: (1)
the non-touch epochs, (a) the 30 ms before touch in the same responses, or (b) 10 ms windows of
exploration time away from touch; (2) the measure at the best post-touch time point, (a)
fraction of touches detected or (b) percent correct, both with the threshold that maximises
percent correct on the training half. Within (1a), after the answer to the question, the
criterion lists three samples: (a1) the window ending at onset (`pre_end`, -10 to 0 ms), (a2)
the earliest (`pre_start`, -30 to -20 ms) and (a3) every pre-touch window pooled (`pre_all`, the
21 windows from -30 to -20 ms through -10 to 0 ms). Reading (1b) uses windows with no touch
onset from 60 ms before to 100 ms after them (a standard choice).
20 repetitions per population size give the mean and the 95% bounds.

In [ ]:
t0 = time.time()
DEC = {}
for g in ("L4C2", "L4out"):
    I_list, Q_list = [], []
    for no in G[g]:
        c = by_no[no]
        d = D[no]
        R = hires.touch_responses(c, d["s"], d["T"])
        Ii = hires.causal_integrate(R.astype(np.int32))
        I_list.append(Ii)
        Q_list.append({"pre_end": Ii[:, 29], "pre_start": Ii[:, 9], "pre_all": Ii[:, 9:30].ravel(), "explore": hires.nontouch_windows(c, d["s"], d["T"], d["E"])})
    DEC[g] = (I_list, Q_list)
    print(g, "touch responses per cell:", [len(x) for x in I_list], "| non-touch windows per cell (1b):", [len(q["explore"]) for q in Q_list])
N_GRID = list(range(1, 21)) + [25, 30, 40, 60, 100, 200]
rng4a = np.random.default_rng(4100)
det_rows = []
for g in ("L4C2", "L4out"):
    I_list, Q_list = DEC[g]
    for N in N_GRID:
        res = hires.touch_detection(I_list, Q_list, N, rng4a, reps=20)
        for rd, arr in res.items():
            for mi, measure in enumerate(("detected", "percent_correct")):
                best = arr[:, :, mi].max(axis=1)
                det_rows.append(dict(group=g, N=N, nontouch=rd, measure=measure, mean=best.mean(),
                                     lo=np.percentile(best, 2.5), hi=np.percentile(best, 97.5)))
DET = pd.DataFrame(det_rows)
print(f"{time.time() - t0:.0f} s")

In [ ]:
d4 = DET[DET.group == "L4C2"]
n95 = d4[d4["mean"] >= 0.95].groupby(["nontouch", "measure"]).N.min().reindex(pd.MultiIndex.from_product([["pre_end", "pre_start", "pre_all", "explore"], ["detected", "percent_correct"]]))
print("smallest N whose mean performance reaches 95%:")
print(n95.to_string())
print("within 15 +- 1.5:", {k: within(v, 15, 1.5) for k, v in n95.items()})
print(d4[d4.N.isin([5, 10, 13, 14, 15, 16, 17, 20, 30])].pivot_table(index="N", columns=["nontouch", "measure"], values="mean").round(3).to_string())

**Result: blocked (methods_underspecified).** The smallest
population whose mean performance reaches 95% depends on the readings: 11 (fraction detected)
or 13 (percent correct) neurons with non-touch windows from exploration time; 17, 30 and 25 with
percent correct and the pre-touch windows (a1), (a2) and (a3); and 4, 1 and 1 with the fraction
of touches detected and the same windows (the threshold that maximises percent correct detects
nearly every touch). None is within 15 +- 1.5. The added reading (a3) changes the random draws
of the others, which moved the exploration and percent-correct value from 11 to 13, the (a1)
value from 18 to 17 and the (a1) fraction-detected value to 4 (1 or 2 before); 20 repetitions per size
leave the mean near 95% uncertain by a few neurons.

In [ ]:
d15 = DET[DET.N == 15].pivot_table(index=["nontouch", "measure"], columns="group", values=["mean", "hi"])
print(d15.round(3).to_string())
for (rd, ms), g in DET.groupby(["nontouch", "measure"]):
    c2_15 = g[(g.group == "L4C2") & (g.N == 15)]["mean"].iloc[0]
    o = g[g.group == "L4out"]
    o15 = o[o.N == 15].iloc[0]
    reach = o[o["mean"] >= c2_15].N.min()
    print(f"{rd:9s} {ms:15s}: N=15 outside {o15['mean']:.3f} (upper bound {o15.hi:.3f}) vs C2 {c2_15:.3f}; below C2 and bound < 0.95: {o15['mean'] < c2_15 and o15.hi < 0.95};"
          f" outside N for C2's N=15 performance: {reach if pd.notna(reach) else '> 200'} (ratio {reach / 15 if pd.notna(reach) else np.inf:.1f})")
fig, ax = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for a_, ms in zip(ax, ("detected", "percent_correct")):
    for g, col in (("L4C2", "red"), ("L4out", "grey")):
        x = DET[(DET.group == g) & (DET.nontouch == "explore") & (DET.measure == ms)]
        a_.plot(x.N, x["mean"], color=col, label=g)
        a_.fill_between(x.N, x.lo, x.hi, color=col, alpha=0.2)
    a_.set_xscale("log"); a_.axhline(0.95, ls="--", c="k", lw=0.5); a_.set_xlabel("neurons"); a_.set_title(f"explore, {ms}", fontsize=8)
ax[0].set_ylabel("performance"); ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("plotted range %.3f to %.3f" % (DET[DET.nontouch == "explore"].lo.min(), DET[DET.nontouch == "explore"].hi.max()))

**Result: blocked (methods_underspecified).** The claim depends
on fig4a-touch-detection-c2. With percent correct, 15 outside neurons perform at 0.53 to 0.55
(upper bound 0.56 to 0.60) against 0.90 to 0.97 in C2, and no outside population up to 200
reaches C2's level, so detection is poor. With the fraction of touches detected, 15 outside
neurons detect 0.91 to 0.94 of touches with an upper bound of 1.000, so the bound does not
exclude 95%. The readings disagree.

## fig4b-touch-time-precision

> A naive Bayes decoder reading 200 L4 C2 neurons integrated over 10 ms decodes the time elapsed
> since touch with high precision, a minimum uncertainty of 0.55 ms (95% confidence) at 10 ms
> after touch onset.

Multinomial naive Bayes over the 10 ms integrated responses of 200 neurons, 100 bootstrap runs.
Open details from the claims file: (1) the uncertainty, (a) RMS error, (b) half width of the
95% bound of decoded time, (c) 1.96 x RMS; (2) training and test sets, (a) halves of 1000 draws
or (b) independent sets of 1000 draws; (3) the times the decoder chooses among, added after the
answer to the question, (c-all) every 1 ms where a full 10 ms window fits, from -21 ms, or
(c-post) every 1 ms from onset on; (4) the smoothing of zero counts, add-one (alpha = 1), with a
smaller pseudocount (alpha = 0.01) as a variant.

In [ ]:
t0 = time.time()
I_c2 = DEC["L4C2"][0]
tb_rows = []
for si, split in enumerate(("half", "independent")):
    for ci, t0b in enumerate((9, 30)):
        for ai, alpha in enumerate((1.0, 0.01)):     # add-one smoothing (criterion), a smaller pseudocount (variant)
            rng4b = np.random.default_rng([4200, si, ci] + ([ai] if ai else []))
            dec, tru, rep = hires.time_decoding(I_c2, rng4b, N=200, reps=100, split=split, t0_bin=t0b, alpha=alpha)
            for t in range(0, 21):
                sel = tru == t
                err = dec[sel] - t
                rms = np.sqrt(np.mean(err ** 2))
                lo, hi = np.percentile(dec[sel], [2.5, 97.5])
                tb_rows.append(dict(split=split, classes_from=t0b - 30, alpha=alpha, t=t, rms=rms, half95=(hi - lo) / 2,
                                    rms196=1.96 * rms, exact=np.mean(err == 0)))
TB = pd.DataFrame(tb_rows)
print(f"{time.time() - t0:.0f} s")


In [ ]:
print(TB[TB.t.isin([6, 8, 9, 10, 11, 12, 14]) & (TB.alpha == 1.0)].round(3).to_string(index=False))
near = TB[(TB.t >= 5) & (TB.t <= 15)]
mins = near.groupby(["alpha", "split", "classes_from"])[["rms", "half95", "rms196"]].min()
print("minimum over 5 to 15 ms after onset:")
print(mins.round(3).to_string())
print("minimum over all times 0 to 20 ms, and where:")
for (al, sp, cf), g in TB.groupby(["alpha", "split", "classes_from"]):
    print(al, sp, cf, {m: (round(g[m].min(), 3), int(g.t[g[m].idxmin()])) for m in ("rms", "half95", "rms196")})
print("within 0.55 +- 0.055:", {k: [m for m in ("rms", "half95", "rms196") if within(v[m], 0.55, 0.055)] for k, v in mins.iterrows()})
rng4bi = np.random.default_rng(4210)
inset = []
for N in (10, 25, 50, 100, 200):
    dec, tru, rep = hires.time_decoding(I_c2, rng4bi, N=N, reps=10, split="half", t0_bin=9)
    sel = tru == 10
    inset.append((N, np.sqrt(np.mean((dec[sel] - 10) ** 2))))
print("RMS error at 10 ms by population size (10 runs each):", [(n, round(e, 2)) for n, e in inset])

**Result: not reproduced.** With add-one smoothing the smallest
uncertainty 5 to 15 ms after onset, at 8 ms after onset, is an RMS error of 1.09 to 1.40 ms, a
half width of the 95% bound of 2.0 ms, or 2.14 to 2.74 ms as 1.96 x RMS, under both splits and
both sets of time classes; none is within 0.55 +- 0.055 ms. The smaller pseudocount, a variant,
gives 1.18 to 1.48 ms RMS and the same 2.0 ms half width. The error falls with population size
(19.9 ms at 10 neurons to 1.55 ms at 200, at 10 ms after onset), as the Figure 4B inset shows.

## fig4c-phase-decoding

> Even 1000 L4 neurons give only a rough estimate of whisking phase, a resolution of 24 deg in
> C2 and 33 deg outside C2 at d' = 1 (76% correct discrimination).

Per cell, spike counts in 120 phase bins for every whisk cycle used for phase tuning, as the
answered criterion says: complete cycles with amplitude above 2.5 deg throughout, monotonic
phase, no lick, not within 70 ms after a touch and with the pole in reach. 1000 neurons drawn
with replacement and circularly shifted to uniformly distributed preferred phases (from the
cosine fit; 0 for the cells without phase spikes); a Poisson naive Bayes decision between two
phases, 50 test samples per phase and phase difference, phase differences of 6 to 180 deg in 6
deg steps, 100 runs, median. Open details from the claims file: (1) responses, (a) counts of
held-out cycles or (b) Poisson counts with the cycle-averaged rates; (2) shifts, (a) random or
(b) even; (3) empty bins, add-one smoothing of the counts, or under (1b) a floor rate, here one
spike over all of the cell's cycles and bins (the paper gives no value). The earlier run's
cycles (every cycle peaking above 2.5 deg) and smoothing (one spike spread over the 120 bins)
are shown as a variant.

In [ ]:
t0 = time.time()
CB, CB_all = {}, {}
for g in ("L4C2", "L4out"):
    Cl, Ca, pref = [], [], []
    for no in G[g]:
        cyc = D[no]["cyc"]
        cp = cyc[cyc.phase_ok & cyc.in_sample]                  # the cycles of phase tuning (criterion)
        Cl.append(hires.cycle_bin_counts(by_no[no], D[no]["s"], cp))
        ca = cyc[cyc.amp_max > hires.WHISK_AMP]                 # every cycle peaking above 2.5 deg (earlier run, variant)
        Ca.append(hires.cycle_bin_counts(by_no[no], D[no]["s"], ca))
        phi = base.loc[no, "phi_pref"]
        pref.append(0.0 if pd.isna(phi) else phi)
    CB[g] = (Cl, np.array(pref))
    CB_all[g] = (Ca, np.array(pref))
    for name, L in (("phase-tuning cycles", Cl), ("all cycles > 2.5 deg (variant)", Ca)):
        print(g, name, "| cycles per cell: median %d, range %d to %d; spikes per cycle, mean over cells %.3f; cells with no spike %d of %d" % (
            np.median([len(x) for x in L]), min(len(x) for x in L), max(len(x) for x in L),
            np.mean([x.sum() / max(len(x), 1) for x in L]), sum(x.sum() == 0 for x in L), len(L)))
READ4C = [("cycles", "add_one"), ("poisson", "add_one"), ("poisson", "floor")]
pd_rows = []
for g in ("L4C2", "L4out"):
    for mi, (model, smooth) in enumerate(READ4C):
        for shi, shifts in enumerate(("random", "even")):
            rng4c = np.random.default_rng([4300, mi, shi, GROUP_ORDER.index(g), 1])
            res, cur = hires.phase_discrimination(CB[g][0], CB[g][1], rng4c, N=1000, reps=100, model=model, shifts=shifts, smooth=smooth)
            pd_rows.append(dict(cycles="phase tuning", group=g, model=model, smooth=smooth, shifts=shifts,
                                median_deg=np.nanmedian(res) if np.isfinite(res).any() else np.nan,
                                runs_reaching=int(np.isfinite(res).sum()), lo=np.nanpercentile(res, 2.5) if np.isfinite(res).any() else np.nan,
                                hi=np.nanpercentile(res, 97.5) if np.isfinite(res).any() else np.nan, pc_at_180=cur[:, -1].mean()))
    # variant: the earlier run's cycles and smoothing, Poisson rates, random shifts
    rng4c = np.random.default_rng([4300, 9, 0, GROUP_ORDER.index(g)])
    res, cur = hires.phase_discrimination(CB_all[g][0], CB_all[g][1], rng4c, N=1000, reps=100, model="poisson", shifts="random", smooth="spread")
    pd_rows.append(dict(cycles="all > 2.5 deg (variant)", group=g, model="poisson", smooth="spread", shifts="random",
                        median_deg=np.nanmedian(res) if np.isfinite(res).any() else np.nan, runs_reaching=int(np.isfinite(res).sum()),
                        lo=np.nanpercentile(res, 2.5) if np.isfinite(res).any() else np.nan,
                        hi=np.nanpercentile(res, 97.5) if np.isfinite(res).any() else np.nan, pc_at_180=cur[:, -1].mean()))
PDC = pd.DataFrame(pd_rows)
print(f"{time.time() - t0:.0f} s")


In [ ]:
print(PDC.round(3).to_string(index=False))
for (cy, model, smooth, shifts), g in PDC.groupby(["cycles", "model", "smooth", "shifts"]):
    a_, b_ = g.set_index("group").loc["L4C2"], g.set_index("group").loc["L4out"]
    print(f"{cy} | {model}/{smooth}/{shifts}: C2 {a_.median_deg:.1f} deg (24 +- 2.4: {within(a_.median_deg, 24, 2.4)}), "
          f"outside {b_.median_deg:.1f} deg (33 +- 3.3: {within(b_.median_deg, 33, 3.3)}), C2 better: {a_.median_deg < b_.median_deg}")

**Result: blocked (methods_underspecified).** No reading gives the
paper's values, and the readings of the empty bins, which the paper does not treat, move the C2
resolution to both sides of 24 deg. With held-out cycle counts and add-one smoothing, neither
group reaches 76% correct in any run. With Poisson rates and add-one smoothing, C2 reaches it
at 84.0 deg (random shifts) or 60.0 deg (even shifts) and outside C2 at 35.7 or 40.7 deg, so C2
is worse than outside. With a floor rate, C2 reaches it at 14.8 or 12.0 deg and outside C2 at
6.0 deg, again better outside. The earlier run's cycles and smoothing, a variant, give 64.5 deg
in C2 and 123.0 deg outside. The phase-tuning cycles are few (median 137 per C2 cell, 0.100
spikes per cycle, 6 of 31 C2 cells with no spike in them), so the decoder depends heavily on how
empty bins are filled.

## fig5c-modulation-index

> Touch response magnitude of L4 C2 neurons is strongly modulated by touch order, whisker
> velocity before touch and maximum curvature during touch, with mean modulation indices of 0.71
> +- 0.25, 0.71 +- 0.27 and 0.72 +- 0.23.

Per cell, touches sorted by the property and split into 10 bins of equal numbers; mean spikes
per touch in the cell's touch window per bin; index (max - min) / (max + min). Open details
from the claims file: velocity at the onset frame or in the frame before; curvature over the
whole touch or 0 to 20 ms; touch order by ranks with ties broken at random or by bins of whole
order values. Every touch-window reading is run.

In [ ]:
def mi_bins(values, counts, n_bins=10, rng=None, whole=False):
    """Mean count per bin of equal numbers of touches sorted by value; returns the bin means."""
    values = np.asarray(values, float)
    if whole:
        u, inv, cnt = np.unique(values, return_inverse=True, return_counts=True)
        mid = (np.cumsum(cnt) - cnt / 2) / len(values)
        b = np.minimum((mid * n_bins).astype(int), n_bins - 1)[inv]
    else:
        key = values + (rng.random(len(values)) * 1e-6 if rng is not None else 0)
        order = np.argsort(key, kind="stable")
        b = np.empty(len(values), int)
        b[order] = np.arange(len(values)) * n_bins // len(values)
    return np.array([counts[b == k].mean() for k in range(n_bins) if np.any(b == k)])


def mod_index(m):
    return (m.max() - m.min()) / (m.max() + m.min()) if m.max() + m.min() > 0 else np.nan


t0 = time.time()
mi_rows, curves = [], []
for no in G["L4C2"]:
    c = by_no[no]
    d = D[no]
    T = d["T"]
    props = {("order", "ranks"): T.order.astype(float), ("order", "whole"): T.order.astype(float),
             ("velocity", "onset"): hires.touch_velocity(c, T, "onset"), ("velocity", "before"): hires.touch_velocity(c, T, "before"),
             ("curvature", "touch"): hires.touch_max_dkappa(c, T, "touch"), ("curvature", "0-20"): hires.touch_max_dkappa(c, T, "0-20")}
    for key, r in W[(W.cell == no) & (W.lat_reading == "a")].set_index(WKEY[1:]).iterrows():
        n = hires.spikes_per_touch(c, d["s"], T, r.latency, r.t_on)
        for (prop, rd), v in props.items():
            rng5 = np.random.default_rng([5100, no])
            m = mi_bins(v, n, rng=rng5, whole=(rd == "whole"))
            mi_rows.append(dict(cell=no, **dict(zip(WKEY[1:], key)), prop=prop, reading=rd, mi=mod_index(m), n_bins=len(m)))
            curves.append(dict(cell=no, **dict(zip(WKEY[1:], key)), prop=prop, reading=rd, curve=m / m.max() if m.max() > 0 else m))
MI = pd.DataFrame(mi_rows)
print(f"{time.time() - t0:.0f} s; bins per curve: {MI.n_bins.value_counts().to_dict()}")

In [ ]:
mis = MI.groupby(WKEY[1:] + ["prop", "reading"]).mi.agg(["mean", "std", "count"]).reset_index()
exp5 = {"order": (0.71, 0.25), "velocity": (0.71, 0.27), "curvature": (0.72, 0.23)}
mis["ok"] = [within(r["mean"], *exp5[r.prop]) for _, r in mis.iterrows()]
print(mis.groupby(["prop", "reading"])["mean"].agg(["min", "max"]).round(3).to_string())
print(mis.groupby(["prop", "reading", "start"]).ok.agg(["sum", "size"]).to_string())
print("range of the means by start of the cutoff search:")
print(mis.groupby(["prop", "start"])["mean"].agg(["min", "max"]).round(3).to_string())
sel = (mis.shuffle == "uniform") & (mis.nearest == "before") & (mis.tmax == 100) & (mis.start == "cross")
print("first reading of each open detail (uniform, before, 100 ms, transition cutoff; ranks, onset, whole touch):",
      {p: round(float(mis[sel & (mis.prop == p) & (mis.reading == r)]["mean"].iloc[0]), 3) for p, r in (("order", "ranks"), ("velocity", "onset"), ("curvature", "touch"))})
print("cells with an undefined index (no spikes in any bin):", MI[MI.mi.isna()].groupby(["prop"]).cell.unique().apply(list).to_dict())

**Result: reproduced.** Under the transition cutoff every reading of
the window, velocity, curvature and touch order gives mean modulation indices within tolerance:
touch order 0.666 to 0.692, velocity 0.703 to 0.735 and curvature 0.664 to 0.698 (paper 0.71 +-
0.25, 0.71 +- 0.27, 0.72 +- 0.23); under the first reading of each detail 0.677, 0.713 and
0.672. The literal cutoff, a variant, gives 0.744 to 0.894, also within tolerance.

## fig5-modulation-correlations

> Across cells, responsiveness to pre-touch velocity and to maximum curvature covary strongly
> (r = 0.80), and responsiveness to touch order is somewhat less correlated with both (r = 0.70
> and 0.65).

Readings: (a) Pearson correlation across the 31 cells of the modulation indices; (b) Pearson
correlation of the peak-normalised 10-bin curves pooled over cells and bins. Under every
reading of fig5c.

In [ ]:
CV = pd.DataFrame(curves)
cor_rows = []
for key, g in MI.groupby(WKEY[1:]):
    for ro in ("ranks", "whole"):
        for rv in ("onset", "before"):
            for rc in ("touch", "0-20"):
                sel = {"order": ro, "velocity": rv, "curvature": rc}
                x = {p: g[(g.prop == p) & (g.reading == sel[p])].set_index("cell").mi for p in sel}
                df = pd.DataFrame(x).dropna()
                cv = CV[(CV.shuffle == key[0]) & (CV.nearest == key[1]) & (CV.tmax == key[2]) & (CV.start == key[3])]
                cc = {p: cv[(cv.prop == p) & (cv.reading == sel[p])].set_index("cell").curve for p in sel}
                common = [k for k in cc["order"].index if all(len(cc[p][k]) == 10 for p in sel)]
                pooled = {p: np.concatenate([cc[p][k] for k in common]) for p in sel}
                for rd, data in (("(a) indices", df), ("(b) curves", pd.DataFrame(pooled).dropna())):
                    r_vc, p_vc = stats.pearsonr(data.velocity, data.curvature)
                    r_ov, p_ov = stats.pearsonr(data.order, data.velocity)
                    r_oc, p_oc = stats.pearsonr(data.order, data.curvature)
                    cor_rows.append(dict(**dict(zip(WKEY[1:], key)), order=ro, velocity=rv, curvature=rc, reading=rd, n=len(data),
                                         r_vc=r_vc, r_ov=r_ov, r_oc=r_oc, p_max=max(p_vc, p_ov, p_oc),
                                         ok=within(r_vc, 0.80, 0.08) and within(r_ov, 0.70, 0.07) and within(r_oc, 0.65, 0.065)
                                         and max(p_vc, p_ov, p_oc) < 0.05 and r_vc >= max(r_ov, r_oc)))
COR = pd.DataFrame(cor_rows)
print(COR.groupby(["reading", "start"])[["r_vc", "r_ov", "r_oc"]].agg(["min", "max"]).round(3).to_string())
print(COR.groupby(["reading", "start"]).ok.agg(["sum", "size"]).to_string())

**Result: blocked (methods_underspecified).** Under reading (a),
correlations of the indices across cells, with the transition cutoff r is 0.618 to 0.788 for
velocity and curvature, 0.706 to 0.788 for order and velocity, and 0.588 to 0.641 for order and
curvature, and the criterion holds in 16 of 64 combinations of the fig5c readings; where it
fails, at least one r is outside its tolerance or velocity and curvature is not the largest. Under reading
(b), pooled curves, r is 0.27 to 0.51 for velocity and curvature and the criterion never holds.
The fig5c readings give different outcomes. The literal cutoff, a variant, holds in 8 of 64
under (a).

## fig6s1b-clusters-less-variable

> The clustering of touches into five groups finds subsets with significantly less variability
> in touch properties than random samples of touches.

Per L4 C2 cell, touches with inter-contact interval > 250 ms; z-scored pre-touch velocity (the
frame before onset, fig5c's "pretouch" reading) and maximum curvature change 0 to 20 ms after
onset; OPTICS ordering; five groups of N touches, N = 20% of the selected touches and at least
20. Open details from the claims file: OPTICS minimum points 5 or N; interval from the previous
offset or onset. The first group's s.d. of z-scored velocity is compared with 1000 random
samples of N of the cell's selected touches.

In [ ]:
t0 = time.time()
GRP = {}
cl_rows = []
for oi, ici_from in enumerate(("offset", "onset")):
    for mi_, minpts in enumerate(("5", "N")):
        for no in G["L4C2"]:
            c = by_no[no]
            T = D[no]["T"]
            sel = np.flatnonzero(hires.ici(T, c, ici_from) > 250)
            v = hires.touch_velocity(c, T, "before")[sel]
            k = hires.touch_max_dkappa(c, T, "0-20")[sel]
            X = np.c_[stats.zscore(v), stats.zscore(k)]
            X = np.nan_to_num(X)
            N = max(20, int(round(0.2 * len(sel))))
            groups = hires.optics_groups(X, N, 5, 5 if minpts == "5" else N) if len(sel) >= N else []
            GRP[(ici_from, minpts, no)] = [sel[g] for g in groups]
            rng6 = np.random.default_rng([6100, oi, mi_, no])
            if groups:
                g1 = X[groups[0], 0].std(ddof=1)
                rnd = np.array([X[rng6.choice(len(X), N, replace=False), 0].std(ddof=1) for _ in range(1000)])
                p5 = np.percentile(rnd, 5)
            else:
                g1, p5 = np.nan, np.nan
            cl_rows.append(dict(ici_from=ici_from, minpts=minpts, cell=no, n_selected=len(sel), N=N, n_groups=len(groups),
                                sd_group1=g1, random_p5=p5, less=bool(g1 < p5) if groups else False))
CL = pd.DataFrame(cl_rows)
print(f"{time.time() - t0:.0f} s")

In [ ]:
print(CL.groupby(["ici_from", "minpts"]).agg(cells_with_groups=("n_groups", lambda x: int((x > 0).sum())), five_groups=("n_groups", lambda x: int((x == 5).sum())),
                                           less=("less", "sum"), n=("less", "size"), median_selected=("n_selected", "median")).to_string())
for key, g in CL.groupby(["ici_from", "minpts"]):
    print(key, "fraction of the 31 cells with the first group below the 5th percentile: %.2f" % g.less.mean())

**Result: reproduced.** Under all four readings, 28 of the 31
L4 C2 cells (0.90) have a first group whose s.d. of z-scored velocity is below the 5th percentile
of 1000 random samples of the same size; 28 or 29 cells have enough touches for at least one
group, and 8 for all five.

## example-cell-ff-around-touch

> For the example neuron, the 10 ms sliding-window Fano factor is about 1 before and after
> touch, where the mean spike count is about 0.03, and falls near the peak of the touch
> response, where the mean count is about 0.7.

Windows of 10 ms slid in 1 ms steps from -50 to +40 ms around onset, across touches with
inter-contact interval > 250 ms (from the previous offset or onset). "Before touch" is read as
the windows that end at or before onset (-50 to -10 ms starts), a standard choice; the peak is
the window of highest mean count. Readings of the touches: (a) pooled, (b) each of the five
groups (every clustering reading). Readings of the cell: the Figure 1 cell, or any L4 C2 cell.

In [ ]:
starts = np.arange(-50, 41)
pre = starts <= -10


def ff_profile(c, s, onsets):
    X = hires.window_counts(c, s, onsets, starts, 10)
    m = X.mean(0)
    v = X.var(0, ddof=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        ff = v / m
    pk = int(np.argmax(m))
    return dict(n=len(onsets), m_pre=m[pre].mean(), ff_pre=np.nanmean(ff[pre]), m_peak=m[pk], ff_peak=ff[pk], peak_start=int(starts[pk]))


rows = []
for no in G["L4C2"]:
    c = by_no[no]
    T = D[no]["T"]
    for ici_from in ("offset", "onset"):
        sel = np.flatnonzero(hires.ici(T, c, ici_from) > 250)
        rows.append(dict(cell=no, touches="pooled", ici_from=ici_from, minpts="-", group=0, **ff_profile(c, D[no]["s"], T.on[sel])))
        for minpts in ("5", "N"):
            for gi, gsel in enumerate(GRP[(ici_from, minpts, no)]):
                rows.append(dict(cell=no, touches="group", ici_from=ici_from, minpts=minpts, group=gi + 1, **ff_profile(c, D[no]["s"], T.on[gsel])))
FFP = pd.DataFrame(rows)
FFP["ok"] = [within(r.m_pre, 0.03, 0.003) and within(r.m_peak, 0.7, 0.07) and abs(r.ff_pre - 1) <= 0.1 and r.ff_peak < 1 for r in FFP.itertuples()]
print(FFP[FFP.cell == EXC].round(3).to_string(index=False))
print("L4 C2 cells matching under pooled touches:", sorted(FFP[(FFP.touches == "pooled") & FFP.ok].cell.unique().tolist()),
      "| under a group:", sorted(FFP[(FFP.touches == "group") & FFP.ok].cell.unique().tolist()))

**Result: blocked (methods_underspecified).** The Figure 1 cell
(23) matches under no reading: its mean 10 ms count before touch is 0.000 to 0.008 (paper about
0.03), its peak mean count 0.15 to 1.36, and its FF at the peak 0.61 to 1.82. With pooled touches
no L4 C2 cell matches; with grouped touches cell 11 does, in 2 of 394 cell and group units. The
two readings of the touches give different outcomes under the criterion's rule for other
cells.

## fig6cd-ff-near-binomial

> Across L4 C2 neurons, the Fano factor following touch drops below one, lying close to the
> binomial limit (the theoretical minimum) for most cells.

Per cell and touch group, the FF in 10 ms windows starting 0 to 20 ms after onset; its minimum
per group (1a) or across groups (1b). "Close": (2a) the binomial limit lies inside the 95%
interval of 1000 resamples of the touches, or (2b) the FF is nearer the binomial limit than 1.
The binomial limit for a mean count m is f(1 - f)/m with f the fractional part of m (1 - m for
m < 1). For (1a) each cell and group counts as one unit.

In [ ]:
starts20 = np.arange(0, 21)
t0 = time.time()
gff_rows = []
for oi, ici_from in enumerate(("offset", "onset")):
    for mi_, minpts in enumerate(("5", "N")):
        for no in G["L4C2"]:
            c = by_no[no]
            T = D[no]["T"]
            for gi, gsel in enumerate(GRP[(ici_from, minpts, no)]):
                X = hires.window_counts(c, D[no]["s"], T.on[gsel], starts20, 10)
                m = X.mean(0)
                with np.errstate(invalid="ignore", divide="ignore"):
                    ff = X.var(0, ddof=1) / m
                j = int(np.nanargmin(ff)) if np.isfinite(ff).any() else 0
                rngb = np.random.default_rng([6300, oi, mi_, no, gi])
                idx = rngb.integers(0, len(X), (1000, len(X)))
                xb = X[:, j][idx]
                with np.errstate(invalid="ignore", divide="ignore"):
                    ffb = xb.var(1, ddof=1) / xb.mean(1)
                lo, hi = np.nanpercentile(ffb, [2.5, 97.5]) if np.isfinite(ffb).any() else (np.nan, np.nan)
                bl = float(hires.binomial_limit(m[j]))
                gff_rows.append(dict(ici_from=ici_from, minpts=minpts, cell=no, group=gi + 1, n=len(gsel), start=int(starts20[j]),
                                     m=m[j], ff=ff[j], binom=bl, lo=lo, hi=hi, ff_by_start=ff, m_by_start=m))
GFF = pd.DataFrame(gff_rows)
GFF["close_a"] = (GFF.binom >= GFF.lo) & (GFF.binom <= GFF.hi)
GFF["close_b"] = (GFF.ff - GFF.binom).abs() < (GFF.ff - 1).abs()
print(f"{time.time() - t0:.0f} s; groups with no spikes in any window (FF undefined): {int(GFF.ff.isna().sum())} of {len(GFF)}")

In [ ]:
rows = []
for (ici_from, minpts), g in GFF.groupby(["ici_from", "minpts"]):
    g = g.dropna(subset=["ff"])
    best = g.loc[g.groupby("cell").ff.idxmin()]
    for which, u in (("(1a) per group", g), ("(1b) across groups", best)):
        for close in ("close_a", "close_b"):
            rows.append(dict(ici_from=ici_from, minpts=minpts, minimum=which, close=close, n_units=len(u), mean_min_ff=u.ff.mean(),
                             fraction_close=u[close].mean(), ok=(u.ff.mean() < 1) and (u[close].mean() > 0.5)))
F6 = pd.DataFrame(rows)
print(F6.round(3).to_string(index=False))
print("combinations meeting the criterion: %d of %d" % (F6.ok.sum(), len(F6)))

**Result: reproduced.** Under all 16 combinations of readings the
population mean of the minimum FF after touch is below 1 (0.504 to 0.676) and the FF is close to
the binomial limit in more than half of the units (0.79 to 1.00).

## fig6d-not-refractory

> The low Fano factor after touch cannot be explained by a Poisson process with a 2.3 ms
> refractory period and rates matched to each neuron.

Per cell, spike trains simulated at 0.1 ms resolution as a Poisson process with an absolute
refractory period of 2.3 ms, one train per real touch; the FF in the same 10 ms windows. After
the answer to the question, the model's average touch response is matched to the data: its
rate is the free rate of Berry and Meister (1998), lambda(t) = r(t) / (1 - integral of r over
[t - 2.3 ms, t)), with r the cell's touch PSTH in 1 ms bins from the same touches, starting 10
ms before onset so that the integral is defined in the first window. The PSTH used directly as
the rate, the earlier run's model, is shown as a variant. Open details from the claims file: (1)
touches pooled (> 250 ms touches) or per group; (2) the comparison, (a) the population mean
data FF below the model's at the window of the data minimum by more than the sum of the two
s.e.m., or (b) the per-cell data FF below the model's in more than half the cells, sign test
p < 0.05.

In [ ]:
def free_rate(rate_ms, dead=2.3, dt=0.1, cap=0.95):
    """Berry and Meister (1998): the rate of the process before the dead time, lambda(t) = r(t) /
    (1 - integral of r over [t - dead, t)), so that the output rate is r. r is given per 1 ms bin
    and spread over steps of dt; the integral is capped at `cap` where it would reach 1."""
    r = np.repeat(rate_ms, int(round(1 / dt)))
    k = int(round(dead / dt))
    c = np.r_[0, np.cumsum(r * dt)]
    i = np.arange(len(r))
    integ = c[i] - c[np.maximum(i - k, 0)]
    return r / (1 - np.minimum(integ, cap)), int((integ > cap).sum())


def simulate_refractory(rate_fine, n_trains, rng, dead=2.3, dt=0.1):
    """Spike times (ms) of n_trains inhomogeneous Poisson trains with an absolute dead time;
    rate_fine is in spikes/ms per step of dt."""
    p = rate_fine * dt
    last = np.full(n_trains, -np.inf)
    out = [[] for _ in range(n_trains)]
    for i, pi in enumerate(p):
        t = i * dt
        fire = (rng.random(n_trains) < pi) & (t - last >= dead)
        last[fire] = t
        for k in np.flatnonzero(fire):
            out[k].append(t)
    return out


def ff_windows(trains, starts, w=10):
    X = np.array([[np.sum((np.asarray(tr) >= s) & (np.asarray(tr) < s + w)) for s in starts] for tr in trains])
    m = X.mean(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        return X.var(0, ddof=1) / m


PRE6D = 10   # ms of PSTH before onset, so that the integral of the free rate starts before the first window
t0 = time.time()
ref_rows = []
for oi, ici_from in enumerate(("offset", "onset")):
    sets = {("pooled", "-"): None}
    for minpts in ("5", "N"):
        sets[("group", minpts)] = minpts
    for (touches, minpts), _ in sets.items():
        for no in G["L4C2"]:
            c = by_no[no]
            T = D[no]["T"]
            s = D[no]["s"]
            units = [np.flatnonzero(hires.ici(T, c, ici_from) > 250)] if touches == "pooled" else GRP[(ici_from, minpts, no)]
            for gi, sel in enumerate(units):
                on = T.on[sel]
                X = hires.window_counts(c, s, on, np.arange(-PRE6D, 31), 1)   # 1 ms PSTH, -10 to 30 ms
                rate = X.mean(0)
                Xd = hires.window_counts(c, s, on, starts20, 10)
                with np.errstate(invalid="ignore", divide="ignore"):
                    ff_data = Xd.var(0, ddof=1) / Xd.mean(0)
                lam, n_cap = free_rate(rate)
                for mi, (model, rf) in enumerate((("compensated", lam), ("uncompensated", np.repeat(rate, 10)))):
                    rngr = np.random.default_rng([6400, oi, 0 if touches == "pooled" else (1 if minpts == "5" else 2), no, gi, mi])
                    sim = [list(np.asarray(tr) - PRE6D) for tr in simulate_refractory(rf, len(on), rngr)]
                    Xm = np.array([[np.sum((np.asarray(tr) >= a) & (np.asarray(tr) < a + 10)) for a in starts20] for tr in sim])
                    ref_rows.append(dict(ici_from=ici_from, touches=touches, minpts=minpts, cell=no, group=gi + 1, model=model, ff_data=ff_data,
                                         ff_model=ff_windows(sim, starts20), mean_data=Xd.mean(0), mean_model=Xm.mean(0), n_capped=n_cap))
REF = pd.DataFrame(ref_rows)
print(f"{time.time() - t0:.0f} s")

In [ ]:
rows = []
for (model, ici_from, touches, minpts), g in REF.groupby(["model", "ici_from", "touches", "minpts"]):
    Dd = np.vstack(g.ff_data.to_numpy())
    Mm = np.vstack(g.ff_model.to_numpy())
    md, mm = np.nanmean(Dd, 0), np.nanmean(Mm, 0)
    sd_ = np.nanstd(Dd, 0, ddof=1) / np.sqrt(np.sum(np.isfinite(Dd), 0))
    sm_ = np.nanstd(Mm, 0, ddof=1) / np.sqrt(np.sum(np.isfinite(Mm), 0))
    j = int(np.nanargmin(md))
    a_ok = (mm[j] - md[j]) > (sd_[j] + sm_[j])
    jc = np.array([np.nanargmin(r) if np.isfinite(r).any() else -1 for r in Dd])
    ok = jc >= 0
    below = np.array([Dd[i, jc[i]] < Mm[i, jc[i]] for i in np.flatnonzero(ok)])
    below = below[np.isfinite(Mm[np.flatnonzero(ok), jc[ok]])]
    k, n = int(below.sum()), len(below)
    p = stats.binomtest(k, n, 0.5, alternative="greater").pvalue if n else np.nan
    rows.append(dict(model=model, ici_from=ici_from, touches=touches, minpts=minpts, units=len(g), start_of_min=int(starts20[j]), data_ff=md[j], model_ff=mm[j],
                     sem_sum=sd_[j] + sm_[j], a_ok=a_ok, cells_below=k, n=n, sign_p=p, b_ok=(k > n / 2) and p < 0.05))
F6D = pd.DataFrame(rows)
print(F6D.round(4).to_string(index=False))
# does the model's mean count match the data's? ratio of the population mean counts in the 10 ms windows
for model, g in REF.groupby("model"):
    md = np.vstack(g.mean_data.to_numpy()).mean(0)
    mm = np.vstack(g.mean_model.to_numpy()).mean(0)
    print(f"{model}: model / data mean count over the 21 windows, {np.min(mm / md):.3f} to {np.max(mm / md):.3f} (at the 0 to 10 ms window {mm[0] / md[0]:.3f});"
          f" steps where the free-rate integral was capped: {int(g.n_capped.sum())}")

**Result: blocked (methods_underspecified).** The compensated model
matches the data's mean count (model over data 1.001 to 1.034 across the windows; 0.80 to 0.89
without compensation) and its FF at the data minimum is 0.71 to 0.79. Under reading (2a) the
population mean data FF (0.75 to 0.82) is not below the model's by more than the sum of the two
s.e.m. in any of the 6 touch readings. Under reading (2b) the per-cell data FF is below the
model's in 4 of the 6 (61 of 89 units, 61 of 100, 59 of 99, and 22 of 31 pooled cells counted
from onsets, sign test p = 0.0003 to 0.035), but not with groups of N points counted from
offsets (52 of 91, p = 0.10) or pooled touches counted from offsets (17 of 31, p = 0.36). The
readings disagree. The uncompensated variant, whose FF is higher, gives (2b) in all 6.

## Figure 6E: fig6e-sample-epoch-ff, fig6e-random-38ms-ff, fig6e-touch-38ms-ff, fig6e-touch-10ms-ff, fig6e-similar-touch-ff

> Counting spikes over the whole sample epoch, ignoring the millisecond structure of behavior,
> gives a very large Fano factor of 6.57 +- 5.41 (mean +- s.d. over L4 C2 cells).

> Counting spikes in random 38 ms windows, as many per trial as there are touches, gives a Fano
> factor of 1.59 +- 0.49.

> Counting spikes in 38 ms windows aligned to touch onset (after a 6 ms latency) gives a Fano
> factor of 1.37 +- 0.74.

> A narrower 10 ms window after touch reduces the Fano factor to 0.73 +- 0.19.

> Computed only across similar touches, the 10 ms Fano factor falls further to 0.54 +- 0.27,
> so that L4 responses to touch are almost noiseless.

In [ ]:
x = ffs[G["L4C2"]]
print(f"sample-epoch FF over the 31 L4 C2 cells: mean {x.mean():.2f}, s.d. {x.std():.2f}, median {x.median():.2f}, range {x.min():.2f} to {x.max():.2f}")
print("within 6.57 +- 5.41 and above 1:", within(x.mean(), 6.57, 5.41) and x.mean() > 1)

**Result: reproduced.** The sample-epoch FF over the 31 L4 C2 cells
is 6.58 +- 5.40 (paper 6.57 +- 5.41).

In [ ]:
rng6e = np.random.default_rng(6500)
rr = {"(a) sample epoch": [], "(b) whole trial": []}
for no in G["L4C2"]:
    c = by_no[no]
    T = D[no]["T"]
    s = np.sort(D[no]["s"])
    ntouch = np.bincount(T.trial, minlength=len(c.trials))
    e = EP_by[no].set_index("trial")
    nseg = np.diff(np.r_[c.seg_start, len(c.vt)])
    for rd in rr:
        ffd = []
        for _ in range(100):
            st = []
            for k in np.flatnonzero(ntouch):
                if rd.startswith("(a)"):
                    if k not in e.index or e.loc[k, "off_ms"] - e.loc[k, "on_ms"] < 38:
                        continue
                    lo, hi = e.loc[k, "on_ms"], e.loc[k, "off_ms"] - 38
                else:
                    lo, hi = 0, nseg[k] - 38
                st.append(k * hires.TRIAL_STRIDE_MS + rng6e.uniform(lo, hi, ntouch[k]))
            st = np.concatenate(st)
            ffd.append(hires.fano(hires.counts_in(s, st, 38)))
        rr[rd].append(np.nanmean(ffd))
for rd, v in rr.items():
    v = np.array(v)
    print(f"{rd}: mean {np.nanmean(v):.2f}, s.d. {np.nanstd(v, ddof=1):.2f}; within 1.59 +- 0.49: {within(np.nanmean(v), 1.59, 0.49)}")
FF2 = {rd: np.nanmean(v) for rd, v in rr.items()}

**Result: reproduced.** Random 38 ms windows give 1.56 +- 0.47 within
the sample epoch and 1.59 +- 0.47 anywhere in the trial (paper 1.59 +- 0.49).

In [ ]:
FF3, FF4 = {}, {}
for no in G["L4C2"]:
    c = by_no[no]
    T = D[no]["T"]
    s = D[no]["s"]
    for tr in ("all", "ici>250 offset", "ici>250 onset"):
        sel = np.arange(len(T.on)) if tr == "all" else np.flatnonzero(hires.ici(T, c, tr.split()[1]) > 250)
        X38 = hires.window_counts(c, s, T.on[sel], np.array([6.0]), 38)[:, 0]
        FF3.setdefault(tr, []).append(hires.fano(X38))
        X10 = hires.window_counts(c, s, T.on[sel], starts20, 10)
        with np.errstate(invalid="ignore", divide="ignore"):
            ff = X10.var(0, ddof=1) / X10.mean(0)
        for rd, lim in (("0-20", 20), ("0-10", 10)):
            v = ff[starts20 <= lim]
            FF4.setdefault((tr, rd), []).append(np.nanmin(v) if np.isfinite(v).any() else np.nan)
for tr, v in FF3.items():
    v = np.array(v)
    print(f"touches {tr}: FF 6-44 ms mean {np.nanmean(v):.2f}, s.d. {np.nanstd(v, ddof=1):.2f}, n {np.isfinite(v).sum()}; within 1.37 +- 0.74: {within(np.nanmean(v), 1.37, 0.74)}")

**Result: reproduced.** Windows 6 to 44 ms after touch give 1.39 +- 0.60
over all touches and 1.34 or 1.36 +- 0.74 over touches with an interval above 250 ms (paper 1.37
+- 0.74).

In [ ]:
for key, v in FF4.items():
    v = np.array(v)
    print(f"touches {key[0]}, window starts {key[1]} ms: minimum FF mean {np.nanmean(v):.2f}, s.d. {np.nanstd(v, ddof=1):.2f}, n {np.isfinite(v).sum()}; within 0.73 +- 0.19: {within(np.nanmean(v), 0.73, 0.19)}")

**Result: reproduced.** The minimum 10 ms FF after touch is 0.85 or 0.86
over all touches and 0.70 to 0.74 over touches with an interval above 250 ms, within 0.73 +- 0.19
under all six readings.

In [ ]:
rows = []
for (ici_from, minpts), g in GFF.groupby(["ici_from", "minpts"]):
    for rd, lim in (("0-20", 20), ("0-10", 10)):
        per = []
        for no, gg in g.groupby("cell"):
            mins = [np.nanmin(f[starts20 <= lim]) if np.isfinite(f[starts20 <= lim]).any() else np.nan for f in gg.ff_by_start]
            per.append((np.nanmin(mins) if np.isfinite(mins).any() else np.nan, np.nanmean(mins)))
        per = np.array(per)
        for vi, val in enumerate(("(a) min across groups", "(b) mean of group minima")):
            v = per[:, vi]
            m5 = np.nanmean(v)
            ff1, ff2 = ffs[G["L4C2"]].mean(), FF2["(a) sample epoch"]
            ff3 = np.nanmean(FF3[f"ici>250 {ici_from}"])
            ff4 = np.nanmean(FF4[(f"ici>250 {ici_from}", rd)])
            rows.append(dict(ici_from=ici_from, minpts=minpts, starts=rd, value=val, mean=m5, sd=np.nanstd(v, ddof=1), n=int(np.isfinite(v).sum()),
                             ok_value=within(m5, 0.54, 0.27), ordered=bool(ff1 > ff2 > ff3 > ff4 > m5), ff1=ff1, ff2=ff2, ff3=ff3, ff4=ff4))
F6E = pd.DataFrame(rows)
F6E["ok"] = F6E.ok_value & F6E.ordered
print(F6E.round(3).to_string(index=False))
print("combinations meeting the criterion: %d of %d" % (F6E.ok.sum(), len(F6E)))
print("order with the whole-trial random windows (2b) and all touches (3a, 4a):",
      [round(v, 2) for v in (ffs[G["L4C2"]].mean(), FF2["(b) whole trial"], np.nanmean(FF3["all"]), np.nanmean(FF4[("all", "0-20")]))])

**Result: reproduced.** Within the touch groups the minimum 10 ms FF
is 0.504 to 0.691 under all 16 readings (paper 0.54 +- 0.27), and the five conditions fall in the
paper's order (6.58 > 1.56 > 1.34 to 1.36 > 0.70 to 0.74 > 0.50 to 0.69) under each.

## whisk-cycle-spikes

> L4 neurons fire few spikes during whisker movements, on average fewer than 0.15 spikes per
> whisk cycle, which hinders decoding of whisking phase.

Spikes per whisk cycle over the cycles used for phase analysis (amplitude > 2.5 deg, monotonic
phase, no lick, not within 70 ms after touch, in exploration time). Readings: L4 C2 or all 41
L4 cells; mean over cells or over all cycles pooled. As a variant, the same cycles at any time
of the trial.

In [ ]:
rows = []
for no in G["L4C2"] + G["L4out"]:
    c = by_no[no]
    cyc = D[no]["cyc"]
    si = np.sort(hires.spike_samples(c, D[no]["s"]))
    for rd, cc in (("exploration", cyc[cyc.phase_ok & cyc.in_sample]), ("any time (variant)", cyc[cyc.phase_ok])):
        n = np.searchsorted(si, cc.stop.to_numpy()) - np.searchsorted(si, cc.start.to_numpy())
        rows.append(dict(cell=no, group=base.loc[no, "group"], cycles=rd, n_cycles=len(cc), spikes=int(n.sum())))
WCS = pd.DataFrame(rows)
WCS["per_cycle"] = WCS.spikes / WCS.n_cycles
for rd, g in WCS.groupby("cycles"):
    for pop, gg in (("(a) L4 C2", g[g.group == "L4C2"]), ("(b) all L4", g)):
        print(f"{rd}, {pop}: mean over cells {gg.per_cycle.mean():.3f}, pooled {gg.spikes.sum() / gg.n_cycles.sum():.3f} spikes/cycle "
              f"({gg.n_cycles.sum()} cycles); below 0.15: {gg.per_cycle.mean() < 0.15}, {gg.spikes.sum() / gg.n_cycles.sum() < 0.15}")
print("cells above 0.15 spikes/cycle (exploration):", WCS[(WCS.cycles == "exploration") & (WCS.per_cycle > 0.15)].cell.tolist())

**Result: reproduced.** L4 cells fire 0.100 (L4 C2, mean over cells) to
0.114 (all L4) spikes per phase-analysis whisk cycle, 0.092 and 0.114 pooled, all below 0.15;
eight cells are above 0.15.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| recordings-counts | reproduced |
| recordings-mice | not reproduced |
| behavior-performance | reproduced |
| behavior-sample-epoch | reproduced |
| behavior-whisking-bouts | partially reproduced |
| behavior-touches-reaction | partially reproduced |
| example-cell-rates | reproduced |
| example-cell-sample-ff | reproduced |
| example-cell-touch-window | partially reproduced |
| example-cell-ff-around-touch | blocked: methods underspecified |
| text-spikes-per-touch | reproduced |
| text-whisking-rates | not reproduced |
| example-cell-phase-modulation | reproduced |
| table1-l4-c2 | reproduced |
| table1-l4-outside | blocked: methods underspecified |
| table1-l5 | partially reproduced |
| latency-determinable | not reproduced |
| l4-outside-weaker-touch | reproduced |
| l5-higher-rates | reproduced |
| phase-mod-l4-inside-outside | reproduced |
| phase-mod-l4-vs-l5 | reproduced |
| whisking-rate-l4-no-change | partially reproduced |
| whisking-rate-l5-increase | reproduced |
| fig3c-touch-spikes-l4c2, fig3c-touch-spikes-outside, fig3c-touch-spikes-l5 | blocked: methods underspecified |
| fig3d-touch-phase-l4c2, fig3d-touch-phase-outside, fig3d-touch-phase-l5, fig3d-phase-only | blocked: methods underspecified |
| fig4a-touch-detection-c2, fig4a-touch-detection-outside | blocked: methods underspecified |
| fig4b-touch-time-precision | not reproduced |
| fig4c-phase-decoding | blocked: methods underspecified |
| fig5c-modulation-index | reproduced |
| fig5-modulation-correlations | blocked: methods underspecified |
| fig6s1b-clusters-less-variable | reproduced |
| fig6cd-ff-near-binomial | reproduced |
| fig6d-not-refractory | blocked: methods underspecified |
| fig6e-sample-epoch-ff, fig6e-random-38ms-ff, fig6e-touch-38ms-ff, fig6e-touch-10ms-ff, fig6e-similar-touch-ff | reproduced |
| whisk-cycle-spikes | reproduced |

Of the 45 claims, 22 are reproduced, 5 partially reproduced, 4 not reproduced and 14 blocked as
methods underspecified. The answers to the seven questions of the first run changed four
outcomes: table1-l4-c2 (blocked to reproduced) and table1-l5 (blocked to partially reproduced)
once the touch-window cutoff is the transition, whisking-rate-l5-increase (partially to
reproduced) once the increase is the median of the paired differences, and fig4c-phase-decoding
(not reproduced to blocked) once the empty-bin readings move its C2 resolution to both sides of
the paper's. The Figure 3 claims stay blocked, now on the range of the cutoff curve (100 or 200
ms), which the paper does not give.

How the analysis differs from the paper's.

- The paper's analysis code is not public (the claims file records that the repository it
  links holds only the contact detection). Every procedure is rebuilt from the claims file.
- Touch onsets and offsets, the Hilbert amplitude and phase, and the lick and pole series are
  the dandiset's own. The stored amplitude and phase match a 6 to 60 Hz fourth-order
  Butterworth band-pass and Hilbert transform of `theta_at_base`.
- Spikes are placed on the `SpikeTrain` timestamps, which agree with the touch response across
  both kinds of file; the voltage timestamps would move 36 files 10 ms later.
- The details asked about in `questions/000013.yaml` (where the cutoff search starts, the
  longest window of the cutoff curve, the pre-touch window of the touch detector, the decoder's
  time classes and smoothing, the phase decoder's cycles and empty bins, the rate of the
  refractory model, the measure of the L5 whisking increase) are now set or listed as readings
  by the claims file and run as it says. The values chosen here where the answers list a
  reading without a number: the floor rate of the phase decoder (one spike over all of a cell's
  cycles and bins) and the smaller pseudocount of the time decoder (0.01). Whisking periods for
  Table 1 are read as the claims file defines them, outside touch epochs. Standard choices that
  are not expected to change a result: the polynomial's conditioning, the guard around touches
  for non-touch windows, the random seeds.
- Fewer repetitions than a reader might expect where the paper does not give a number: 20 per
  population size for touch detection.

What was found about the dataset.

- It holds all 52 recordings with the paper's group labels (31, 10, 11) and 115 +- 60 trials per
  neuron; the performance, the sample epoch and the Figure 6E Fano factors come out as the
  paper's to two or three digits, and the example cell of Figure 1 can be identified (cell 23).
- The files fall in batches converted differently. In 36 files the `SpikeTrain` timestamps run
  10 ms before the voltage timestamps of the same samples. In 38 files (those 36 and cells 20
  and 40) `theta_filt` is not the band-passed angle the stored amplitude and phase come from,
  and the voltage, with its stated conversion of 0.001 to volts, spans only microvolts. In the
  other 14 files none of these holds.
- The subjects (23) and genotypes do not match the paper's 21 mice and four lines: three
  subjects (six cells) have no genotype, and the C57BL6 and Ai32 x PV-Cre counts differ by one
  cell each.
- `pole_position` is described in mm, but its values range over several scales from file to
  file: 0.2 to 1.6 in some, 992 to 13888 (motor steps) in others, about 1e8 in others, 0 and
  5952 in one, and up to 1.4e12 in another.
- `set_point` is described as the Hilbert amplitude but follows the angle minus its band-passed
  part (r 0.957 to 0.999 where it has no missing values; 23 files hold missing values).
- File names say `ogen`, but no file holds an optogenetic stimulus and `stim_present` is 0 in
  every trial.
- `first_lick_time` disagrees with the lick series in some trials (printed in the data check).

In [ ]:
print(f"notebook run time: {(time.time() - T_START) / 60:.1f} min")